# LTE Downlink Throughput — Final Pre-Anomaly Feature Engineering

This is the production-ready preprocessing notebook for the LTE downlink throughput anomaly-detection pipeline.

It performs a clean generated-output reset, validates the one-second sampling cadence, builds leakage-safe telecom features, preserves past-only causal event context for ML, and exports the final pre-anomaly and modeling tables used by Notebook 11.


## Run order and output contract

1. Run this notebook using **Restart Kernel → Run All**.
2. Confirm the leakage and sampling validation summaries pass.
3. Run `11. LTE_Throughput_Anomaly_Detection_Final_Handoff_Ready.ipynb`.

Generated outputs are written under `Throughput_Anomaly_Detection_Outputs/`. The raw `Raw_Data/` folder is never deleted.


In [1]:
# =========================
# 1. Imports and settings
# =========================

from pathlib import Path
import re
import json
import warnings
import shutil
import hashlib
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True
except Exception as exc:
    go = None
    px = None
    make_subplots = None
    PLOTLY_AVAILABLE = False
    print("Plotly is unavailable; interactive visual exports will be skipped.")
    print("Reason:", exc)

from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 150)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 260)

# Matplotlib remains imported only as a lightweight fallback.
# The preferred visualization backend is Plotly for interactive HTML outputs.
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

# -------------------------
# Project paths
# -------------------------
DATA_DIR = Path("Raw_Data")
DRIVE_TEST_PATH = DATA_DIR / "Network_Drive_Test.pkl"
OUTPUT_ROOT = Path("Throughput_Anomaly_Detection_Outputs")

# Production-run freshness control.
# Run Notebook 01 first. It removes generated outputs only; Raw_Data is never touched.
CLEAN_GENERATED_OUTPUTS_BEFORE_RUN = True
EXPECTED_OUTPUT_ROOT_NAME = "Throughput_Anomaly_Detection_Outputs"
if CLEAN_GENERATED_OUTPUTS_BEFORE_RUN:
    safe_root = OUTPUT_ROOT.resolve()
    if OUTPUT_ROOT.name != EXPECTED_OUTPUT_ROOT_NAME or OUTPUT_ROOT.is_symlink():
        raise RuntimeError(f"Refusing to clean unexpected or symbolic output path: {safe_root}")
    if OUTPUT_ROOT.exists():
        shutil.rmtree(OUTPUT_ROOT)
        print("Removed generated output tree from the previous run:", safe_root)

PREPROCESSING_DIR = OUTPUT_ROOT / "01_preprocessing_cleaning"
ANOMALY_DIR = OUTPUT_ROOT / "02_anomaly_detection"
PLOTS_DIR = OUTPUT_ROOT / "03_plots"
RCA_HANDOFF_DIR = OUTPUT_ROOT / "04_rca_handoff_final"
RCA_KEY_FILES_DIR = RCA_HANDOFF_DIR / "00_key_review_files"
# OUTPUT_DIR remains as the root for backward compatibility inside helper functions.
OUTPUT_DIR = OUTPUT_ROOT
for _dir in [OUTPUT_ROOT, PREPROCESSING_DIR, ANOMALY_DIR, PLOTS_DIR, RCA_HANDOFF_DIR, RCA_KEY_FILES_DIR]:
    _dir.mkdir(parents=True, exist_ok=True)

# -------------------------
# Cleaning configuration
# -------------------------
MISSING_THRESHOLD_PCT = 70.0
DROP_FULLY_MISSING_PROTECTED_COLUMNS = True
RANDOM_SEED = 42
PROFILE_SAMPLE_ROWS = 100_000
EARTH_RADIUS_KM = 6371.0088
# Project-verified sampling cadence: exactly one sample per second.
SAMPLE_INTERVAL_SECONDS = 1.0

# -------------------------
# Throughput configuration
# -------------------------
# Most common drive-test exports store the selected LTE throughput KPI in Mbps.
# If your exported target is Kbps, set THROUGHPUT_TO_MBPS = 1 / 1000.
THROUGHPUT_TO_MBPS = 1.0

LOW_TP_FIXED_MBPS = 10.0
MIN_EXPECTED_TP_MBPS_FOR_UNDERPERFORMANCE = 25.0
UNDERPERFORMANCE_RATIO_THRESHOLD = 0.35
UNDERPERFORMANCE_GAP_MBPS = 15.0

# Robust lower-tail z-score setting.
# A classic modified-z threshold of -3.5 can become too strict after log1p() compression
# and may produce 0 rows. -3.0 keeps this as an extreme lower-tail detector.
ROBUST_Z_LOWER_THRESHOLD = -3.0

# Group P75 expected-throughput settings.
MIN_GROUP_SIZE_FOR_P75 = 30
P75_QUANTILE = 0.75

# Rolling/sudden-drop settings.
ROLLING_WINDOW_SAMPLES = 11
ROLLING_DROP_RATIO_THRESHOLD = 0.50
SUDDEN_DROP_PREV_MIN_MBPS = 10.0
SUDDEN_DROP_RATIO_THRESHOLD = 0.50
SUDDEN_DROP_GAP_MBPS = 5.0

# Sustained-window and bad-session settings.
# Sudden-drop detection looks for contrast versus a healthy local baseline.
# The following thresholds catch the different case where the whole local window/session is bad.
MIN_ROLLING_SAMPLES = 5
VERY_LOW_TP_MBPS = 5.0
ROLLING_BAD_WINDOW_MEDIAN_MBPS = LOW_TP_FIXED_MBPS
ROLLING_BAD_WINDOW_P25_MBPS = VERY_LOW_TP_MBPS

SESSION_MIN_ROWS_FOR_BAD_SESSION = 20
BAD_SESSION_MEDIAN_MBPS = LOW_TP_FIXED_MBPS
BAD_SESSION_P25_MBPS = VERY_LOW_TP_MBPS
BAD_SESSION_LOW_TP_RATIO_THRESHOLD = 0.40
SESSION_P10_MAX_THRESHOLD_MBPS = 20.0

# -------------------------
# Download activity-window settings
# -------------------------
# LTE-DL session rows can include script/activity context such as HTTP/HTTPS download windows.
# Earlier versions used active download windows as the main analysis mask. The latest V1 uses
# all valid LTE-DL rows because the observed throughput distributions across active/warmup/outside
# activity windows were very close. Activity windows are now kept as context only.
BUILD_DOWNLOAD_ACTIVITY_CONTEXT_FLAGS = True
DOWNLOAD_ACTIVITY_NAME_PATTERN = r"HTTP|HTTPS|Download"
DOWNLOAD_ACTIVITY_WARMUP_SECONDS = 5
DOWNLOAD_ACTIVITY_COOLDOWN_SECONDS = 2

# Visual context settings.
MAX_TOP_CONTEXT_WINDOWS = 8
MAX_CONTEXT_WINDOWS_BY_TYPE = 8
ANOMALY_CONTEXT_SAMPLES_BEFORE = 60
ANOMALY_CONTEXT_SAMPLES_AFTER = 60


# -------------------------
# LTE radio-quality bin thresholds
# -------------------------
# These thresholds are intentionally calibrated for this real drive-test export.
# They are slightly more forgiving than some textbook/paper thresholds because field
# measurements, mobility, server/application behavior, and route conditions can make
# practical throughput lower than ideal lab expectations.
#
# RSRP bins:
#   Excellent: >= -90 dBm
#   Good:      [-100, -90) dBm
#   Fair:      [-120, -100) dBm
#   Poor:      < -120 dBm
RSRP_EXCELLENT_MIN_DBM = -90.0
RSRP_GOOD_MIN_DBM = -100.0
RSRP_FAIR_MIN_DBM = -120.0

# RSRQ bins:
#   Excellent: >= -10 dB
#   Good:      [-15, -10) dB
#   Fair:      [-20, -15) dB
#   Poor:      < -20 dB
RSRQ_EXCELLENT_MIN_DB = -10.0
RSRQ_GOOD_MIN_DB = -15.0
RSRQ_FAIR_MIN_DB = -20.0

# SINR bins:
#   Excellent: >= 15 dB
#   Good:      [10, 15) dB
#   Fair:      [0, 10) dB
#   Poor:      < 0 dB
SINR_EXCELLENT_MIN_DB = 15.0
SINR_GOOD_MIN_DB = 10.0
SINR_FAIR_MIN_DB = 0.0


# -------------------------
# Final visual-QC and false-positive-control settings
# -------------------------
# Visualization clipping only affects plots, never the data used by anomaly detection.
ROUTE_MAP_COLOR_CLIP_Q = 0.95
HISTOGRAM_COLOR_CLIP_Q = 0.99

# RB reliability checks. RB fields are only used as strong evidence when coverage and
# cardinality show that the export contains enough valid RB data.
MIN_RELIABLE_RB_NON_NULL_PCT = 70.0
MIN_RELIABLE_RB_UNIQUE_VALUES = 10
LOW_RB_QUANTILE = 0.25
MEDIUM_RB_QUANTILE = 0.50
HIGH_RB_QUANTILE = 0.75

# Demand/RB evidence controls.
# If RB data is reliable, strict expected-throughput and CA-underperformance flags
# require medium-or-high RB usage. Otherwise low observed throughput can be caused by
# low offered traffic or a scheduler/resource-allocation state, so it should be lower
# confidence instead of a strong capacity anomaly.
REQUIRE_RB_DEMAND_FOR_STRICT_P75 = True
REQUIRE_RB_DEMAND_FOR_CA_UNDERPERFORMANCE = True

# Score calibration after removing aggressive flag stacking.
# These floors prevent visually obvious severe samples from receiving unrealistically low
# scores, while caps prevent low-RB/low-demand rows from looking like confirmed capacity failures.
MIN_SCORE_ANY_TRIGGER = 15.0
MIN_SCORE_FIXED_LOW_TP = 25.0
MIN_SCORE_SUSTAINED_LOW_WINDOW = 35.0
MIN_SCORE_RADIO_LIMITED = 35.0
MIN_SCORE_LOCAL_DROP = 50.0
MIN_SCORE_P75_UNDERPERF = 55.0
MIN_SCORE_SEVERE_P75_UNDERPERF = 75.0
LOW_RB_DEMAND_SCORE_CAP = 45.0

# Score saturation control. Scores are softly capped below 100 unless a truly catastrophic
# case is present. This prevents too many rows from saturating at 100.
SCORE_SOFT_CAP_NON_CATASTROPHIC = 95.0
CATASTROPHIC_ACTUAL_TP_MBPS = 1.0
CATASTROPHIC_EXPECTED_TP_MBPS = 50.0

# RB-efficiency anomaly settings. Throughput per RB becomes useful after RB availability
# was proven reliable. It checks whether throughput is inefficient even when resources exist.
RB_EFFICIENCY_REFERENCE_QUANTILE = 0.10
RB_EFFICIENCY_MIN_ACTUAL_TP_FOR_CONTEXT_MBPS = 0.0


# Continuous score calibration. The final score is no longer just a sum of flags or only
# a set of floors. It combines: strongest-trigger base, continuous anomaly magnitude,
# independent trigger-family bonus, and capped evidence/context bonus.
MULTI_TRIGGER_BONUS_PER_LOG_STEP = 5.0
MULTI_TRIGGER_BONUS_CAP = 12.0
EVIDENCE_BONUS_CAP = 10.0
MAGNITUDE_GAP_DENOMINATOR_MBPS = 80.0
MAGNITUDE_DROP_DENOMINATOR_MBPS = 40.0
MAGNITUDE_LOW_TP_REFERENCE_MBPS = 20.0

# RB-conditioned P75 expected-throughput controls.
USE_RB_CONDITIONED_P75 = True
RB_P75_LABEL = "rb_conditioned"

# P75 reliability controls. Specific group references are more trustworthy than global fallback.
MAX_RELIABLE_P75_FALLBACK_LEVEL = 4
MIN_RELIABLE_P75_GROUP_N = MIN_GROUP_SIZE_FOR_P75

# Bootstrap confidence interval settings for group P75 references.
# These CIs are used as an additional confidence guard for strict P75 underperformance.
P75_CI_BOOTSTRAP_N = 80
P75_CI_ALPHA = 0.05

# Tighter CA-underperformance controls.
CA_UNDERPERF_MIN_EXPECTED_MBPS = 30.0
CA_UNDERPERF_MAX_ACTUAL_MBPS = 15.0
CA_UNDERPERF_RATIO_THRESHOLD = 0.35

# Episode grouping converts consecutive row-level anomalies into reviewable periods.
MAX_SECONDS_GAP_SAME_ANOMALY_EPISODE = 3.0

# Event-window settings used only to prepare anomaly evidence, not final RCA.
# Measurement reports are separated from actual HO execution/failure so mobility context does
# not become too broad. Legacy aliases are created later for compatibility.
EVENT_WINDOWS_SECONDS = {
    "any_event": 30,
    "measurement_report_event": 5,
    "handover_execution": 5,
    "handover_failure": 10,
    "rach_attempt": 10,
    "rach_failure": 10,
    "ca_activation_change": 10,
    "failure_other": 10,
}

# Candidate-vs-handoff split. The candidate table keeps broad statistical findings;
# the baseline handoff table is a stricter subset for practical RCA review.
RCA_HANDOFF_MIN_SCORE = 35.0

print("Notebook configured.")
print("Raw drive-test path:", DRIVE_TEST_PATH)
print("Output root:", OUTPUT_ROOT.resolve())
print("Preprocessing/cleaning folder:", PREPROCESSING_DIR.resolve())
print("Anomaly-detection folder:", ANOMALY_DIR.resolve())
print("Plots folder:", PLOTS_DIR.resolve())
print("Final RCA handoff folder:", RCA_HANDOFF_DIR.resolve())
print("Missing threshold:", MISSING_THRESHOLD_PCT, "%")

Removed generated output tree from the previous run: D:\ITI 9 Months\Graduation Project\Throughput_Anomaly_Detection_Outputs
Notebook configured.
Raw drive-test path: Raw_Data\Network_Drive_Test.pkl
Output root: D:\ITI 9 Months\Graduation Project\Throughput_Anomaly_Detection_Outputs
Preprocessing/cleaning folder: D:\ITI 9 Months\Graduation Project\Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning
Anomaly-detection folder: D:\ITI 9 Months\Graduation Project\Throughput_Anomaly_Detection_Outputs\02_anomaly_detection
Plots folder: D:\ITI 9 Months\Graduation Project\Throughput_Anomaly_Detection_Outputs\03_plots
Final RCA handoff folder: D:\ITI 9 Months\Graduation Project\Throughput_Anomaly_Detection_Outputs\04_rca_handoff_final
Missing threshold: 70.0 %


In [2]:
# =========================
# 2. General utility functions
# =========================

def human_bytes(num_bytes: float) -> str:
    """Convert bytes to a readable memory-size string."""
    if pd.isna(num_bytes):
        return "N/A"
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(num_bytes)
    for unit in units:
        if size < 1024:
            return f"{size:,.2f} {unit}"
        size /= 1024
    return f"{size:,.2f} PB"



def infer_output_dir(filename: str, category: Optional[str] = None) -> Path:
    """
    Route output artifacts into the requested folder.

    category can be:
    - "preprocessing" / "cleaning"
    - "anomaly"
    - "plots"
    - "rca" / "handoff"

    When category is omitted, the filename is classified using stable prefixes.
    """
    if category is not None:
        cat = str(category).lower()
        if cat in {"preprocessing", "cleaning", "preprocess", "clean"}:
            return PREPROCESSING_DIR
        if cat in {"anomaly", "anomalies", "detection"}:
            return ANOMALY_DIR
        if cat in {"plot", "plots", "figure", "figures"}:
            return PLOTS_DIR
        if cat in {"rca", "handoff", "final", "key"}:
            return RCA_HANDOFF_DIR

    name = str(filename).lower()
    preprocessing_prefixes = (
        "raw_", "column_family_", "missing_", "kept_", "removed_",
        "cleaning_", "cleaned_", "drive_clean_", "activity_", "events_",
        "event_", "canonical_", "detected_", "lte_dl_modeling_table",
        "download_activity_", "throughput_by_download_context", "throughput_by_ca_condition",
        "throughput_by_rb_demand", "rb_usage_quality", "threshold_calibration",
    )
    if name.startswith(preprocessing_prefixes):
        return PREPROCESSING_DIR
    return ANOMALY_DIR


def save_table(df: pd.DataFrame, filename: str, index: bool = False, category: Optional[str] = None) -> Path:
    """Save a table as UTF-8 CSV inside the correct output subfolder."""
    folder = infer_output_dir(filename, category=category)
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / filename
    df.to_csv(path, index=index, encoding="utf-8-sig")
    print(f"Saved CSV: {path}  shape={df.shape}")
    return path


def save_parquet_or_csv(df: pd.DataFrame, filename_no_ext: str, index: bool = False, category: Optional[str] = None) -> Path:
    """
    Save a DataFrame as Parquet when pyarrow/fastparquet is available.
    If Parquet support is missing, save a compressed CSV fallback.
    """
    folder = infer_output_dir(filename_no_ext, category=category)
    folder.mkdir(parents=True, exist_ok=True)
    parquet_path = folder / f"{filename_no_ext}.parquet"
    csv_path = folder / f"{filename_no_ext}.csv.gz"
    try:
        df.to_parquet(parquet_path, index=index)
        print(f"Saved Parquet: {parquet_path}  shape={df.shape}")
        return parquet_path
    except Exception as exc:
        print("Parquet export failed. Falling back to compressed CSV.")
        print("Reason:", exc)
        df.to_csv(csv_path, index=index, compression="gzip", encoding="utf-8-sig")
        print(f"Saved compressed CSV: {csv_path}  shape={df.shape}")
        return csv_path


def sample_df(df: pd.DataFrame, n: int = 100_000, seed: int = RANDOM_SEED) -> pd.DataFrame:
    """Return a safe sample. If df is smaller than n, return a copy of the full df."""
    if len(df) <= n:
        return df.copy()
    return df.sample(n=n, random_state=seed).copy()


def safe_value_counts(s: pd.Series, top_n: int = 10) -> Dict[str, int]:
    """Return top value counts as a dict without failing on mixed dtypes."""
    non_null = s.dropna()
    if non_null.empty:
        return {}
    return non_null.astype(str).value_counts().head(top_n).to_dict()


def get_column_family(col: str) -> str:
    """Infer a practical telecom family from a column name."""
    text = str(col)
    if " - " in text:
        return text.split(" - ")[0].strip()
    if "." in text:
        return text.split(".")[0].strip()
    return text.split()[0].strip() if text.split() else "Unknown"


def first_existing(df: pd.DataFrame, candidates: Sequence[str]) -> Optional[str]:
    """Return the first existing column from a candidate list."""
    for col in candidates:
        if col in df.columns:
            return col
    return None


def to_numeric_series(df: pd.DataFrame, col: Optional[str]) -> pd.Series:
    """Return a numeric Series for col; returns all-NaN if col is missing."""
    if col is None or col not in df.columns:
        return pd.Series(np.nan, index=df.index, dtype="float64")
    return pd.to_numeric(df[col], errors="coerce")


def overview(df: pd.DataFrame, name: str) -> None:
    """Display a compact overview of a dataframe."""
    print(f"===== {name} =====")
    print("Shape:", df.shape)
    print("Rows:", f"{len(df):,}")
    print("Columns:", f"{df.shape[1]:,}")
    print("Memory:", human_bytes(df.memory_usage(deep=True).sum()))
    display(pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null": df.notna().sum().values,
        "missing": df.isna().sum().values,
        "missing_pct": (df.isna().mean().values * 100).round(3),
    }).head(250))

In [3]:

# =========================
# 2B. Visualization helper functions
# =========================


def _plotly_output_paths(filename: Optional[str]) -> Tuple[Optional[Path], Optional[Path]]:
    """Resolve interactive HTML and PNG companion paths without losing subfolders."""
    if not filename:
        return None, None
    base = Path(filename)
    output_folder = base.parent if base.parent != Path(".") else PLOTS_DIR
    output_folder.mkdir(parents=True, exist_ok=True)
    html_path = output_folder / f"{base.stem}.html"
    png_path = output_folder / f"{base.stem}.png"
    return html_path, png_path


def save_current_plot(filename: str) -> None:
    """Matplotlib fallback saver used only by any remaining static plots."""
    path = PLOTS_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=140, bbox_inches="tight")
    print(f"Saved static plot: {path}")


def save_plotly_figure(fig, filename: Optional[str], width: int = 1100, height: int = 550) -> None:
    if filename is None:
        fig.show()
        return
    html_path, png_path = _plotly_output_paths(filename)
    fig.write_html(str(html_path), include_plotlyjs="cdn", full_html=True)
    print(f"Saved interactive plot: {html_path}")
    try:
        fig.write_image(str(png_path), width=width, height=height, scale=2)
        print(f"Saved static companion image: {png_path}")
    except Exception as exc:
        print(f"Static Plotly image export skipped for {png_path.name}: {exc}")


def clip_for_plot(s: pd.Series, lower: Optional[float] = None, upper_q: Optional[float] = 0.995) -> pd.Series:
    """Return a numeric series clipped only for visualization, never for modeling."""
    out = pd.to_numeric(s, errors="coerce")
    if lower is not None:
        out = out.clip(lower=lower)
    if upper_q is not None and out.notna().any():
        upper = out.quantile(upper_q)
        out = out.clip(upper=upper)
    return out


def plot_distribution_with_lines(
    df: pd.DataFrame,
    col: str,
    title: str,
    xlabel: str,
    thresholds: Optional[Dict[str, float]] = None,
    bins: int = 80,
    filename: Optional[str] = None,
    clip_upper_q: Optional[float] = 0.995,
) -> None:
    if col not in df.columns:
        print(f"Skipped {title}: column not found: {col}")
        return
    s = clip_for_plot(
        df[col],
        lower=0 if "throughput" in col.lower() or "score" in col.lower() else None,
        upper_q=clip_upper_q,
    ).dropna()
    if s.empty:
        print(f"Skipped {title}: no numeric values in {col}")
        return
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=s, nbinsx=bins, name="Rows", opacity=0.85))
    if thresholds:
        ymax = max(1, int(np.ceil(len(s) / max(1, bins) * 3)))
        for label, value in thresholds.items():
            if pd.notna(value):
                fig.add_vline(x=float(value), line_dash="dash", annotation_text=f"{label}: {float(value):.2f}", annotation_position="top")
    fig.update_layout(title=title, xaxis_title=xlabel, yaxis_title="Rows", bargap=0.05, template="plotly_white", height=480)
    save_plotly_figure(fig, filename, height=480)
    fig.show()


def plot_box_by_category(
    df: pd.DataFrame,
    category_col: str,
    value_col: str,
    title: str,
    ylabel: str,
    filename: Optional[str] = None,
    category_order: Optional[List[Any]] = None,
    clip_upper_q: Optional[float] = 0.995,
) -> None:
    if category_col not in df.columns or value_col not in df.columns:
        print(f"Skipped {title}: missing {category_col} or {value_col}")
        return
    tmp = df[[category_col, value_col]].copy()
    tmp[value_col] = clip_for_plot(tmp[value_col], lower=0, upper_q=clip_upper_q)
    tmp = tmp.dropna(subset=[category_col, value_col])
    if tmp.empty:
        print(f"Skipped {title}: no data after dropping missing values")
        return
    if category_order is None:
        category_order = list(tmp[category_col].astype(str).value_counts().index)
    tmp[category_col] = tmp[category_col].astype(str)
    tmp = tmp[tmp[category_col].isin([str(c) for c in category_order])]
    fig = go.Figure()
    for cat in category_order:
        part = tmp.loc[tmp[category_col] == str(cat), value_col]
        if len(part):
            fig.add_trace(go.Box(y=part, name=str(cat), boxpoints=False))
    fig.update_layout(title=title, xaxis_title=category_col, yaxis_title=ylabel, template="plotly_white", height=520)
    save_plotly_figure(fig, filename, height=520)
    fig.show()


def plot_scatter_sample(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    title: str,
    xlabel: str,
    ylabel: str,
    filename: Optional[str] = None,
    max_points: int = 20_000,
    y_clip_upper_q: Optional[float] = 0.995,
) -> None:
    if x_col not in df.columns or y_col not in df.columns:
        print(f"Skipped {title}: missing {x_col} or {y_col}")
        return
    tmp = df[[x_col, y_col]].copy()
    tmp[y_col] = clip_for_plot(tmp[y_col], lower=0, upper_q=y_clip_upper_q)
    tmp = tmp.dropna()
    if tmp.empty:
        print(f"Skipped {title}: no data")
        return
    tmp = sample_df(tmp, max_points)
    fig = go.Figure(go.Scattergl(x=tmp[x_col], y=tmp[y_col], mode="markers", marker=dict(size=5, opacity=0.35), name="Samples"))
    fig.update_layout(title=title, xaxis_title=xlabel, yaxis_title=ylabel, template="plotly_white", height=500)
    save_plotly_figure(fig, filename, height=500)
    fig.show()


def plot_route_metric(
    df: pd.DataFrame,
    metric_col: str,
    title: str,
    filename: Optional[str] = None,
    max_points: int = 30_000,
    clip_upper_q: Optional[float] = ROUTE_MAP_COLOR_CLIP_Q,
    colorbar_label: Optional[str] = None,
) -> None:
    required = {"latitude", "longitude", metric_col}
    if not required.issubset(df.columns):
        print(f"Skipped {title}: missing one of {required}")
        return
    tmp = df.dropna(subset=["latitude", "longitude", metric_col]).copy()
    if tmp.empty:
        print(f"Skipped {title}: no valid GPS/metric rows")
        return
    tmp = sample_df(tmp, max_points)
    color_values = pd.to_numeric(tmp[metric_col], errors="coerce")
    cb_label = colorbar_label or metric_col
    if clip_upper_q is not None and color_values.notna().any():
        upper = color_values.quantile(clip_upper_q)
        color_values = color_values.clip(upper=upper)
        cb_label = f"{cb_label} (display clipped at P{int(clip_upper_q * 100)}={upper:.2f})"
    fig = go.Figure(go.Scattergl(
        x=tmp["longitude"], y=tmp["latitude"], mode="markers",
        marker=dict(size=6, opacity=0.8, color=color_values, colorscale="Viridis", colorbar=dict(title=cb_label)),
        name=metric_col,
        text=[f"{metric_col}: {v:.3f}" if pd.notna(v) else metric_col for v in color_values],
        hovertemplate="Lon=%{x:.6f}<br>Lat=%{y:.6f}<br>%{text}<extra></extra>",
    ))
    fig.update_layout(title=title, xaxis_title="Longitude", yaxis_title="Latitude", template="plotly_white", height=700)
    save_plotly_figure(fig, filename, height=700)
    fig.show()


def plot_route_anomaly_overlay(
    df: pd.DataFrame,
    filename: Optional[str] = None,
    max_points: int = 40_000,
) -> None:
    required = {"latitude", "longitude", "is_throughput_anomaly"}
    if not required.issubset(df.columns):
        print(f"Skipped anomaly overlay map: missing one of {required}")
        return
    tmp = df.dropna(subset=["latitude", "longitude"]).copy()
    if tmp.empty:
        print("Skipped anomaly overlay map: no valid GPS rows")
        return
    tmp = sample_df(tmp, max_points)
    normal = tmp[~tmp["is_throughput_anomaly"].fillna(False).astype(bool)]
    anom = tmp[tmp["is_throughput_anomaly"].fillna(False).astype(bool)]
    fig = go.Figure()
    if not normal.empty:
        fig.add_trace(go.Scattergl(x=normal["longitude"], y=normal["latitude"], mode="markers", marker=dict(size=5, opacity=0.18), name="Not flagged", hovertemplate="Lon=%{x:.6f}<br>Lat=%{y:.6f}<extra></extra>"))
    if not anom.empty:
        score = pd.to_numeric(anom.get("anomaly_score_0_100", pd.Series(1, index=anom.index)), errors="coerce").fillna(1)
        fig.add_trace(go.Scattergl(
            x=anom["longitude"], y=anom["latitude"], mode="markers",
            marker=dict(size=8, opacity=0.9, color=score, colorscale="Turbo", colorbar=dict(title="Anomaly score (0-100)")),
            name="Flagged anomaly",
            hovertemplate="Lon=%{x:.6f}<br>Lat=%{y:.6f}<br>Score=%{marker.color:.2f}<extra></extra>",
        ))
    fig.update_layout(title="Route with Throughput Anomaly Overlay", xaxis_title="Longitude", yaxis_title="Latitude", template="plotly_white", height=700)
    save_plotly_figure(fig, filename, height=700)
    fig.show()


def plot_flag_percentage_bars(summary_df: pd.DataFrame, filename: Optional[str] = None) -> None:
    if summary_df is None or summary_df.empty or "method_or_case" not in summary_df.columns:
        print("Skipped anomaly method/case flagging-rate plot: summary table is empty")
        return
    tmp = summary_df.copy()
    tmp["flagged_pct"] = pd.to_numeric(tmp.get("flagged_pct", np.nan), errors="coerce")
    tmp = tmp.dropna(subset=["method_or_case", "flagged_pct"]).sort_values("flagged_pct", ascending=True)
    if tmp.empty:
        print("Skipped anomaly method/case flagging-rate plot: no numeric flagged_pct values")
        return
    fig = go.Figure(go.Bar(x=tmp["flagged_pct"], y=tmp["method_or_case"].astype(str), orientation="h", text=tmp["flagged_pct"].round(2), textposition="outside"))
    fig.update_layout(title="Anomaly Method / Case Flagging Rate", xaxis_title="Flagged rows (%)", yaxis_title="Method / case", template="plotly_white", height=max(450, 28 * len(tmp) + 120))
    save_plotly_figure(fig, filename, height=max(450, 28 * len(tmp) + 120))
    fig.show()


def plot_actual_vs_expected(
    df: pd.DataFrame,
    expected_col: str = "expected_tp_p75",
    filename: Optional[str] = None,
    max_points: int = 20_000,
) -> None:
    actual_col = "actual_lte_dl_throughput"
    required = {actual_col, expected_col}
    if not required.issubset(df.columns):
        print(f"Skipped actual-vs-expected plot: missing one of {required}")
        return
    tmp = df[[actual_col, expected_col]].dropna().copy()
    if tmp.empty:
        print("Skipped actual-vs-expected plot: no valid rows")
        return
    tmp = sample_df(tmp, max_points)
    limit = float(max(tmp[expected_col].quantile(0.995), tmp[actual_col].quantile(0.995), 1))
    fig = go.Figure()
    fig.add_trace(go.Scattergl(x=tmp[expected_col], y=tmp[actual_col], mode="markers", marker=dict(size=5, opacity=0.35), name="Samples"))
    fig.add_trace(go.Scatter(x=[0, limit], y=[0, limit], mode="lines", line=dict(dash="dash"), name="Actual = expected"))
    fig.update_layout(title="Actual vs Group-P75 Expected Throughput", xaxis_title="Expected throughput P75 (Mbps)", yaxis_title="Actual LTE DL throughput (Mbps)", template="plotly_white", height=560)
    save_plotly_figure(fig, filename, height=560)
    fig.show()


def plot_anomaly_context(
    df: pd.DataFrame,
    anomaly_df: Optional[pd.DataFrame] = None,
    rank: int = 0,
    samples_before: int = 40,
    samples_after: int = 40,
    filename_prefix: Optional[str] = None,
    title_prefix: str = "Top anomaly context window",
) -> None:
    if anomaly_df is None:
        if "is_throughput_anomaly" not in df.columns:
            print("Skipped anomaly context: no anomaly flag available")
            return
        anomaly_df = df[df["is_throughput_anomaly"].fillna(False).astype(bool)].copy()
    if anomaly_df.empty:
        print("Skipped anomaly context: no anomalies available")
        return
    if "anomaly_score_0_100" in anomaly_df.columns:
        anomaly_df = anomaly_df.sort_values("anomaly_score_0_100", ascending=False)
    if rank >= len(anomaly_df):
        print(f"Skipped anomaly context: rank {rank} out of range")
        return

    row = anomaly_df.iloc[rank]
    sid = row.get("id", None)
    if "id" in df.columns and pd.notna(sid):
        session = df[df["id"] == sid].copy()
    else:
        session = df.copy()
    if "timestamp" in session.columns:
        session = session.sort_values("timestamp")

    center_pos = None
    if "source_index" in row.index and "source_index" in session.columns:
        matches = np.where(session["source_index"].values == row["source_index"])[0]
        if len(matches):
            center_pos = int(matches[0])
    if center_pos is None and "timestamp" in row.index and "timestamp" in session.columns:
        session_ts_utc = pd.to_datetime(session["timestamp"], utc=True, errors="coerce")
        row_ts_utc = pd.to_datetime(row["timestamp"], utc=True, errors="coerce")
        diffs = (session_ts_utc - row_ts_utc).abs()
        center_pos = int(diffs.argmin()) if len(diffs) else None
    if center_pos is None:
        print("Skipped anomaly context: could not locate selected anomaly in session")
        return

    start = max(0, center_pos - samples_before)
    end = min(len(session), center_pos + samples_after + 1)
    ctx = session.iloc[start:end].copy().reset_index(drop=True)
    ctx["context_sample_number"] = np.arange(len(ctx))
    center_x = center_pos - start

    selected_timestamp_text = ""
    if "timestamp" in ctx.columns:
        ts = pd.to_datetime(ctx["timestamp"], utc=True, errors="coerce")
        center_ts = ts.iloc[center_x]
        ctx["seconds_from_selected_anomaly"] = (ts - center_ts).dt.total_seconds()
        if pd.notna(center_ts):
            selected_timestamp_text = f" | selected time: {center_ts}"

    summary_cols = [
        "timestamp", "id", "anomaly_type", "anomaly_type_flags", "anomaly_score_0_100",
        "actual_lte_dl_throughput", "expected_tp_p75", "expected_tp_rb_conditioned_p75", "throughput_ratio_p75", "throughput_ratio_rb_p75",
        "rb_usage_value", "rb_usage_bucket", "rb_demand_confidence", "throughput_per_rb",
        "lte_rsrp", "lte_rsrq", "lte_sinr", "carrier_count",
        "download_context_label", "download_analysis_window_flag",
    ]
    available_summary_cols = [c for c in summary_cols if c in row.index]
    if available_summary_cols:
        print("Selected anomaly summary:")
        display(pd.DataFrame([row[available_summary_cols].to_dict()]))

    hover_seconds = ctx.get("seconds_from_selected_anomaly", pd.Series(np.nan, index=ctx.index))
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12, subplot_titles=("Throughput behavior", "Radio evidence"))
    def _add_line(series_col, name, row_id, dash=None):
        if series_col not in ctx.columns:
            return
        y = pd.to_numeric(ctx[series_col], errors="coerce")
        if not y.notna().any():
            return
        fig.add_trace(go.Scatter(
            x=ctx["context_sample_number"], y=y, mode="lines+markers", name=name,
            line=dict(dash=dash) if dash else {},
            customdata=np.c_[hover_seconds.fillna(np.nan)],
            hovertemplate="Sample=%{x}<br>Seconds from anomaly=%{customdata[0]:+.0f}s<br>" + name + "=%{y:.3f}<extra></extra>",
        ), row=row_id, col=1)

    _add_line("actual_lte_dl_throughput", "Actual TP", 1)
    _add_line("expected_tp_p75", "Radio P75", 1, "dash")
    _add_line("expected_tp_rb_conditioned_p75", "RB-conditioned P75", 1, "dashdot")
    _add_line("rolling_median_tp", "Rolling median", 1, "dot")
    for col in ["lte_rsrp", "lte_rsrq", "lte_sinr"]:
        _add_line(col, col, 2)
    for rr in [1, 2]:
        fig.add_vline(x=center_x, line_dash="dash", row=rr, col=1)
    fig.update_layout(title=f"{title_prefix} #{rank + 1}{selected_timestamp_text}", template="plotly_white", height=850, margin=dict(t=100, r=40, b=50, l=50))
    fig.update_xaxes(title_text="Context sample number", row=2, col=1)
    fig.update_yaxes(title_text="Throughput (Mbps)", row=1, col=1)
    fig.update_yaxes(title_text="Radio KPI value", row=2, col=1)
    filename = f"{filename_prefix}_rank_{rank + 1}_context.html" if filename_prefix else None
    save_plotly_figure(fig, filename, height=850)
    fig.show()


def plot_context_examples_by_type(
    df: pd.DataFrame,
    anomaly_df: pd.DataFrame,
    max_types: int = 8,
    samples_before: int = 60,
    samples_after: int = 60,
    filename_prefix: str = "19_type_example_context",
) -> None:
    if anomaly_df is None or anomaly_df.empty or "anomaly_type" not in anomaly_df.columns:
        print("Skipped type examples: anomaly table is empty or missing anomaly_type")
        return
    types = anomaly_df["anomaly_type"].value_counts().head(max_types).index.tolist()
    for i, anomaly_type in enumerate(types, start=1):
        subset = anomaly_df[anomaly_df["anomaly_type"] == anomaly_type].copy()
        if subset.empty:
            continue
        subset = subset.sort_values("anomaly_score_0_100", ascending=False).head(1)
        print(f"\nRepresentative anomaly type example: {anomaly_type}")
        plot_anomaly_context(
            df,
            anomaly_df=subset,
            rank=0,
            samples_before=samples_before,
            samples_after=samples_after,
            filename_prefix=f"{filename_prefix}_{i}_{str(anomaly_type).replace('/', '_').replace(' ', '_')}",
            title_prefix=f"Representative {anomaly_type}",
        )


In [4]:
# =========================
# 3. Data profiling and cleaning helpers
# =========================

def build_data_dictionary(
    df: pd.DataFrame,
    name: str,
    sample_rows: int = PROFILE_SAMPLE_ROWS,
    top_n: int = 5,
) -> pd.DataFrame:
    """
    Build a practical data dictionary.

    Full missingness is calculated on all rows.
    Examples, unique counts, and numeric summaries are calculated on a sample for speed.
    """
    sample = sample_df(df, sample_rows)
    missing_count = df.isna().sum()
    missing_pct = df.isna().mean() * 100
    non_null_count = df.notna().sum()

    records = []
    for col in df.columns:
        s_full = df[col]
        s = sample[col]
        dtype = str(s_full.dtype)
        family = get_column_family(col)
        nunique_sample = s.nunique(dropna=True)
        non_null_sample = s.notna().sum()

        record = {
            "dataset": name,
            "column": col,
            "family": family,
            "dtype": dtype,
            "full_non_null_count": int(non_null_count[col]),
            "full_missing_count": int(missing_count[col]),
            "full_missing_pct": round(float(missing_pct[col]), 3),
            "sample_unique_count": int(nunique_sample),
            "sample_unique_pct": round(float(nunique_sample / max(non_null_sample, 1) * 100), 3),
            "sample_non_null_examples": s.dropna().astype(str).head(top_n).tolist(),
        }

        if pd.api.types.is_numeric_dtype(s_full):
            numeric_s = pd.to_numeric(s, errors="coerce")
            record.update({
                "sample_min": numeric_s.min(),
                "sample_max": numeric_s.max(),
                "sample_mean": numeric_s.mean(),
                "sample_std": numeric_s.std(),
                "sample_top_values": None,
            })
        else:
            record.update({
                "sample_min": None,
                "sample_max": None,
                "sample_mean": None,
                "sample_std": None,
                "sample_top_values": safe_value_counts(s, top_n=top_n),
            })
        records.append(record)

    return (
        pd.DataFrame(records)
        .sort_values(["full_missing_pct", "family", "column"], ascending=[False, True, True])
        .reset_index(drop=True)
    )


def summarize_column_families(profile: pd.DataFrame) -> pd.DataFrame:
    """Summarize missingness by column family."""
    summary = (
        profile.groupby("family")
        .agg(
            columns=("column", "count"),
            avg_missing_pct=("full_missing_pct", "mean"),
            min_missing_pct=("full_missing_pct", "min"),
            max_missing_pct=("full_missing_pct", "max"),
            zero_missing_columns=("full_missing_pct", lambda x: int((x == 0).sum())),
            fully_missing_columns=("full_missing_pct", lambda x: int((x == 100).sum())),
            above_threshold_missing=("full_missing_pct", lambda x: int((x > MISSING_THRESHOLD_PCT).sum())),
        )
        .reset_index()
        .sort_values(["columns", "avg_missing_pct"], ascending=[False, False])
    )
    summary["avg_missing_pct"] = summary["avg_missing_pct"].round(3)
    return summary


def threshold_impact(profile: pd.DataFrame, thresholds: Sequence[int] = (50, 60, 70, 75, 80, 90, 95, 99)) -> pd.DataFrame:
    """Show how many columns would be removed at each missing-value threshold."""
    total = len(profile)
    rows = []
    for threshold in thresholds:
        removed = int((profile["full_missing_pct"] > threshold).sum())
        rows.append({
            "missing_threshold_pct": threshold,
            "columns_removed_if_not_protected": removed,
            "columns_kept_if_not_protected": total - removed,
            "removed_pct_of_all_columns": round(removed / max(total, 1) * 100, 2),
        })
    return pd.DataFrame(rows)


# Exact columns known from the cleaning outputs and reports.
EXACT_PROTECTED_COLUMNS = {
    "id",
    "imsi",
    "timestamp",
    "Location - Latitude",
    "Location - Longitude",
    "Location - Speed",
    "Location - Altitude",
    "Location - Accuracy",
    "Activity.Start TimeStamp",
    "Activity.End TimeStamp",
    "Activity.Activity",
    "Activity.Status",
    "Activity.Tech",
    "Activity.Duration",
    "Activity.Throughput",
    "Activity.Power",
    "Activity.Quality",
    "Activity.SINR",
    "Events.Timestamp",
    "Events.category",
    "Events.Tech",
    "Events.titles",
    "Events.description",
    "Events.subject",
    "LTE.LTE_Data_KPI.Agg_Throughput_DL",
    "LTE.DownlinkMeasurements.Throughput_DL",
    "LTE - PrimaryCell - PCell_Throughput",
    "LTE - SecondaryCell 1 - SCell1_Throughput",
    "Throughput.HTTP_DL",
    "Throughput.Physical_DL",
}

# Pattern protection preserves sparse columns that are useful for anomaly handoff and later RCA.
PROTECTED_PATTERNS = [
    r"^Location",
    r"^Events\.",
    r"^Activity\.",
    r"^LTE - Serving -",
    r"^LTE - PrimaryCell",
    r"^LTE - PrimaryCell Radio",
    r"^LTE - SecondaryCell 1",
    r"^LTE\.LTE_Data_KPI\.(Agg_Throughput_DL|Agg_Throughput_UL|Agg_RB_DL|Agg_RB_UL|Agg_Bandwidth_DL)$",
    r"^LTE\.DownlinkMeasurements\.Throughput_DL$",
    r"^LTE\.DedicatedRadioLink\.CarrierCount$",
    r"^LTE\.RACH\.",
    r"^Throughput\.(HTTP_DL|Physical_DL)$",
    r"^Extra_KPIs\.(LTE Handover Interruption Time|UMTS Handover Interruption Time|ERRC Connection Setup Time|RRC Connection Setup Time|Attach Delay)$",
]


def is_protected_column(col: str) -> bool:
    """Return True when a column should survive the missingness threshold for throughput/anomaly handoff."""
    text = str(col)
    if text in EXACT_PROTECTED_COLUMNS:
        return True
    return any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in PROTECTED_PATTERNS)

In [5]:
# =========================
# 4. Time, ID, event, and geo helpers
# =========================

def parse_drive_test_id(id_value: Any) -> Dict[str, Any]:
    """
    Parse IDs like:
    542020041221088+20250825121610+Fiji-Nadi-MO-D!20250825121533
    """
    text = str(id_value) if pd.notna(id_value) else ""
    result = {
        "id_imsi": np.nan,
        "id_session_time_raw": np.nan,
        "id_route_test_label": np.nan,
        "id_log_time_raw": np.nan,
        "id_session_time": pd.NaT,
        "id_log_time": pd.NaT,
        "id_location_label": np.nan,
        "id_test_type": np.nan,
    }
    if not text:
        return result

    main_part, bang, log_part = text.partition("!")
    plus_parts = main_part.split("+")

    if len(plus_parts) >= 1:
        result["id_imsi"] = plus_parts[0]
    if len(plus_parts) >= 2:
        result["id_session_time_raw"] = plus_parts[1]
        result["id_session_time"] = pd.to_datetime(plus_parts[1], format="%Y%m%d%H%M%S", errors="coerce")
    if len(plus_parts) >= 3:
        label = "+".join(plus_parts[2:])
        result["id_route_test_label"] = label
        label_parts = label.split("-")
        if len(label_parts) >= 2:
            result["id_location_label"] = "-".join(label_parts[:2])
        if len(label_parts) >= 4:
            result["id_test_type"] = "-".join(label_parts[-2:])
        elif len(label_parts) >= 1:
            result["id_test_type"] = label_parts[-1]
    if bang:
        result["id_log_time_raw"] = log_part
        result["id_log_time"] = pd.to_datetime(log_part, format="%Y%m%d%H%M%S", errors="coerce")

    return result


def add_id_features(df: pd.DataFrame, id_col: str = "id") -> pd.DataFrame:
    """Add parsed route/session/test fields from the drive-test id."""
    if id_col not in df.columns:
        print("ID column not found. ID-derived fields will be missing.")
        return df.copy()
    unique_ids = df[id_col].drop_duplicates()
    parsed = unique_ids.apply(parse_drive_test_id).apply(pd.Series)
    parsed[id_col] = unique_ids.values
    return df.merge(parsed, on=id_col, how="left")


def split_multi_value(value: Any, sep: str = "$$$$") -> List[Any]:
    """Split packed event fields that contain multiple values separated by $$$$."""
    if pd.isna(value):
        return []
    return str(value).split(sep)


def build_events_long(df: pd.DataFrame, id_col: str = "id", sample_time_col: str = "timestamp") -> pd.DataFrame:
    """
    Convert packed row-level Events.* columns into a clean event-level table.
    One output row equals one event.
    """
    event_cols = [
        "Events.Timestamp",
        "Events.category",
        "Events.Tech",
        "Events.titles",
        "Events.description",
        "Events.subject",
    ]
    available = [c for c in event_cols if c in df.columns]
    if not available:
        print("No Events.* columns found.")
        return pd.DataFrame()

    source = df[df[available].notna().any(axis=1)].copy()
    event_rows = []

    for idx, row in source.iterrows():
        split_values = {c: split_multi_value(row[c]) for c in available}
        max_len = max([len(v) for v in split_values.values()] + [0])
        for order in range(max_len):
            rec = {"source_index": idx, "event_order_in_sample": order}
            if id_col in df.columns:
                rec[id_col] = row[id_col]
            if sample_time_col in df.columns:
                rec["sample_timestamp"] = row[sample_time_col]
            for c in available:
                values = split_values[c]
                rec[c] = values[order] if order < len(values) else np.nan
            event_rows.append(rec)

    events = pd.DataFrame(event_rows)
    if events.empty:
        return events

    if "Events.Timestamp" in events.columns:
        events["event_timestamp"] = pd.to_datetime(events["Events.Timestamp"], errors="coerce")
    if "sample_timestamp" in events.columns:
        events["sample_timestamp"] = pd.to_datetime(events["sample_timestamp"], errors="coerce")
        if "event_timestamp" in events.columns:
            events["event_delay_from_sample_seconds"] = (
                events["event_timestamp"] - events["sample_timestamp"]
            ).dt.total_seconds()
    return events


def classify_event_family(row: pd.Series) -> str:
    """
    Create a compact event family for anomaly evidence.

    This is not final RCA. The goal is to keep frequent measurement reports separate
    from actual handover execution/failure, RACH failures, and CA activation changes.
    """
    text = " ".join([
        str(row.get("Events.titles", "")),
        str(row.get("Events.subject", "")),
        str(row.get("Events.description", "")),
    ]).lower()

    # RACH / random access.
    if re.search(r"rach|random access", text):
        if re.search(r"fail|failure|timeout|reject|abort|unsuccess", text):
            return "rach_failure"
        return "rach_attempt"

    # Handover failures and radio-link/mobility failures should be separated from
    # ordinary measurement-report events.
    if re.search(r"handover.*(fail|failure|timeout|reject|abort)|ho.*(fail|failure)|radio link failure|rlf|re[- ]?establishment", text):
        return "handover_failure"

    # Actual mobility execution/context. This is still context only, but it is much
    # less broad than treating A1/A2/A3/A5/A6 measurement reports as handovers.
    if re.search(r"handover command|handover complete|handover success|ho command|ho complete|serving cell change|cell reselection", text):
        return "handover_execution"

    # Measurement reports are common and should not dominate event-related evidence.
    if re.search(r"event a1|event a2|event a3|event a5|event a6|measurement report|meas report", text):
        return "measurement_report_event"

    # Carrier aggregation and SCell state changes.
    if re.search(r"lte ca|carrier aggregation|scell|secondary cell|ca init|ca complete|ca deact|ca activ|activation|deactivation|add scell|release scell", text):
        return "ca_activation_change"

    if re.search(r"fail|failure|timeout|reject|abort|unsuccess", text):
        return "failure_other"
    return "other"


def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized haversine distance in kilometers."""
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return EARTH_RADIUS_KM * 2 * np.arcsin(np.sqrt(a))

# Load raw drive-test data

The next cell is the only required external data input. It expects the original pickle file. Pickle files should only be loaded from trusted sources.

In [6]:
# =========================
# 5. Load the raw drive-test pickle
# =========================

if not DRIVE_TEST_PATH.exists():
    raise FileNotFoundError(
        f"Raw drive-test file was not found at {DRIVE_TEST_PATH}. "
        "Place Network_Drive_Test.pkl inside Raw_Data/ and rerun."
    )

drive_raw_obj = pd.read_pickle(DRIVE_TEST_PATH)

# Some pickle exports may store a dict of tables instead of a direct DataFrame.
# If that happens, select the largest DataFrame as the drive-test table.
if isinstance(drive_raw_obj, pd.DataFrame):
    drive_raw = drive_raw_obj.copy()
elif isinstance(drive_raw_obj, dict):
    dataframe_items = {k: v for k, v in drive_raw_obj.items() if isinstance(v, pd.DataFrame)}
    if not dataframe_items:
        raise ValueError("The pickle contains a dict, but no pandas DataFrame values were found.")
    selected_key = max(dataframe_items, key=lambda k: dataframe_items[k].shape[0] * dataframe_items[k].shape[1])
    print(f"Pickle contains multiple tables. Selected largest DataFrame key: {selected_key}")
    drive_raw = dataframe_items[selected_key].copy()
else:
    raise TypeError(f"Unsupported pickle object type: {type(drive_raw_obj)}")

overview(drive_raw, "Raw drive-test data")

===== Raw drive-test data =====
Shape: (64949, 1365)
Rows: 64,949
Columns: 1,365
Memory: 983.28 MB


,column,dtype,non_null,missing,missing_pct
0,id,object,64949,0,0.000
1,timestamp,"datetime64[ns, UTC+02:00]",64949,0,0.000
2,imsi,int64,64949,0,0.000
3,LTE - PrimaryCell - PCell_CQI,float64,37341,27608,42.507
4,LTE - PrimaryCell - PCell_PCI,float64,37588,27361,42.127
...,...,...,...,...,...
245,Extra_KPIs.LTE Handover Interruption Time,float64,2214,62735,96.591
246,Extra_KPIs.UMTS Handover Interruption Time,float64,3826,61123,94.109
247,Extra_KPIs.MO Call Setup Time Until Connect,float64,254,64695,99.609
248,LTE.RACH.RACH_Reason,object,25766,39183,60.329


# Raw profiling and cleaning plan

The cleaning strategy is controlled and audit-friendly:

1. Build a raw data dictionary.
2. Apply the 70% missing-value threshold.
3. Preserve important time, location, LTE, activity, event, and selected sparse throughput/RCA-handoff fields.
4. Drop 100% missing columns unless they are explicitly needed and configured otherwise.
5. Avoid global numeric imputation because missing telecom measurements often have technical meaning.

In [7]:
# =========================
# 6. Profile raw data and build cleaning plan
# =========================

raw_profile = build_data_dictionary(drive_raw, name="drive_raw")
raw_family_summary = summarize_column_families(raw_profile)
missing_impact = threshold_impact(raw_profile)

save_table(raw_profile, "raw_data_dictionary.csv")
save_table(raw_family_summary, "column_family_summary_before_cleaning.csv")
save_table(missing_impact, "missing_threshold_impact.csv")

print("Raw family summary:")
display(raw_family_summary)
print("Missing-threshold impact:")
display(missing_impact)

# Mark protected columns.
raw_profile["is_protected"] = raw_profile["column"].apply(is_protected_column)

# Reason priority: 100% missing > above threshold > constant/single unique.
raw_profile["remove_reason"] = ""
fully_missing = raw_profile["full_missing_pct"] == 100
above_threshold = raw_profile["full_missing_pct"] > MISSING_THRESHOLD_PCT
constant = raw_profile["sample_unique_count"] <= 1
protected = raw_profile["is_protected"]

if DROP_FULLY_MISSING_PROTECTED_COLUMNS:
    raw_profile.loc[fully_missing, "remove_reason"] = "100% missing"
else:
    raw_profile.loc[fully_missing & (~protected), "remove_reason"] = "100% missing"

raw_profile.loc[(raw_profile["remove_reason"] == "") & above_threshold & (~protected), "remove_reason"] = f">{MISSING_THRESHOLD_PCT:.0f}% missing"
raw_profile.loc[(raw_profile["remove_reason"] == "") & constant & (~protected), "remove_reason"] = "constant / single unique value"

removed_columns_report = raw_profile[raw_profile["remove_reason"] != ""].copy()
kept_columns_report = raw_profile[raw_profile["remove_reason"] == ""].copy()
protected_columns_report = raw_profile[raw_profile["is_protected"]].copy()

save_table(removed_columns_report, "removed_columns_report.csv")
save_table(kept_columns_report, "kept_columns_after_cleaning.csv")
save_table(protected_columns_report, "protected_columns_report.csv")

reason_summary = removed_columns_report["remove_reason"].value_counts().reset_index()
reason_summary.columns = ["remove_reason", "columns_removed"]
save_table(reason_summary, "removed_columns_reason_summary.csv")

display(reason_summary)
print("Kept columns:", len(kept_columns_report))
print("Removed columns:", len(removed_columns_report))
print("Protected columns kept or reviewed:", len(protected_columns_report))

Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\raw_data_dictionary.csv  shape=(1365, 15)
Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\column_family_summary_before_cleaning.csv  shape=(17, 8)
Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\missing_threshold_impact.csv  shape=(8, 4)
Raw family summary:


,family,columns,avg_missing_pct,min_missing_pct,max_missing_pct,zero_missing_columns,fully_missing_columns,above_threshold_missing
5,LTE,440,83.654,41.725,100.000,0,29,317
7,NR5G,302,100.000,100.000,100.000,0,302,302
4,GSM,271,98.468,95.335,100.000,0,119,271
12,WCDMA,230,88.117,59.969,100.000,0,16,176
0,Activity,35,99.452,99.284,100.000,0,4,35
9,SpeedTest,21,100.000,100.000,100.000,0,21,21
3,Extra_KPIs,20,95.964,65.130,100.000,0,8,18
8,PCAP,10,0.000,0.000,0.000,10,0,0
13,YouTube,9,100.000,100.000,100.000,0,9,9
2,Events,6,60.505,56.826,69.816,0,0,0


Missing-threshold impact:


,missing_threshold_pct,columns_removed_if_not_protected,columns_kept_if_not_protected,removed_pct_of_all_columns
0,50,1278,87,93.63
1,60,1256,109,92.01
2,70,1162,203,85.13
3,75,1145,220,83.88
4,80,1129,236,82.71
5,90,1103,262,80.81
6,95,1064,301,77.95
7,99,840,525,61.54


Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\removed_columns_report.csv  shape=(1145, 17)
Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\kept_columns_after_cleaning.csv  shape=(220, 17)
Saved CSV: Throughput_Anomaly_Detection_Outputs\02_anomaly_detection\protected_columns_report.csv  shape=(154, 17)
Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\removed_columns_reason_summary.csv  shape=(3, 2)


,remove_reason,columns_removed
0,>70% missing,601
1,100% missing,518
2,constant / single unique value,26


Kept columns: 220
Removed columns: 1145
Protected columns kept or reviewed: 154


In [8]:
# =========================
# 7. Apply cleaning and derive time/location features
# =========================

kept_columns = kept_columns_report["column"].tolist()
drive_clean = drive_raw[kept_columns].copy()

# Parse timestamp. The row-level timestamp is the main clock for route/event/anomaly alignment.
if "timestamp" in drive_clean.columns:
    drive_clean["timestamp"] = pd.to_datetime(drive_clean["timestamp"], errors="coerce")
else:
    raise KeyError("Required row-level timestamp column 'timestamp' was not found after cleaning.")

# Parse ID/session metadata.
drive_clean = add_id_features(drive_clean, id_col="id") if "id" in drive_clean.columns else drive_clean

# Sort rows to recover session sequence.
sort_cols = [c for c in ["id", "timestamp"] if c in drive_clean.columns]
if sort_cols:
    drive_clean = drive_clean.sort_values(sort_cols).reset_index(drop=True)

# Derived time features.
drive_clean["date"] = drive_clean["timestamp"].dt.date
drive_clean["hour"] = drive_clean["timestamp"].dt.hour
drive_clean["minute"] = drive_clean["timestamp"].dt.minute
if "id" in drive_clean.columns:
    drive_clean["sample_gap_seconds"] = drive_clean.groupby("id")["timestamp"].diff().dt.total_seconds()
    drive_clean["session_elapsed_seconds"] = (
        drive_clean["timestamp"] - drive_clean.groupby("id")["timestamp"].transform("min")
    ).dt.total_seconds()
else:
    drive_clean["sample_gap_seconds"] = drive_clean["timestamp"].diff().dt.total_seconds()
    drive_clean["session_elapsed_seconds"] = (drive_clean["timestamp"] - drive_clean["timestamp"].min()).dt.total_seconds()

# Parse activity timestamps if present.
for col in ["Activity.Start TimeStamp", "Activity.End TimeStamp"]:
    if col in drive_clean.columns:
        drive_clean[col + "_parsed"] = pd.to_datetime(drive_clean[col], errors="coerce")

if {"Activity.Start TimeStamp_parsed", "Activity.End TimeStamp_parsed"}.issubset(drive_clean.columns):
    drive_clean["activity_duration_seconds_from_timestamps"] = (
        drive_clean["Activity.End TimeStamp_parsed"] - drive_clean["Activity.Start TimeStamp_parsed"]
    ).dt.total_seconds()

# Location cleanup and route distance.
LAT_COL = "Location - Latitude"
LON_COL = "Location - Longitude"
SPEED_COL = "Location - Speed"
ACCURACY_COL = "Location - Accuracy"
ALTITUDE_COL = "Location - Altitude"

if {LAT_COL, LON_COL}.issubset(drive_clean.columns):
    drive_clean[LAT_COL] = pd.to_numeric(drive_clean[LAT_COL], errors="coerce")
    drive_clean[LON_COL] = pd.to_numeric(drive_clean[LON_COL], errors="coerce")
    drive_clean["has_valid_location"] = drive_clean[[LAT_COL, LON_COL]].notna().all(axis=1)
    if "id" in drive_clean.columns:
        drive_clean["prev_lat"] = drive_clean.groupby("id")[LAT_COL].shift(1)
        drive_clean["prev_lon"] = drive_clean.groupby("id")[LON_COL].shift(1)
    else:
        drive_clean["prev_lat"] = drive_clean[LAT_COL].shift(1)
        drive_clean["prev_lon"] = drive_clean[LON_COL].shift(1)
    drive_clean["distance_from_prev_m"] = haversine_km(
        drive_clean["prev_lat"], drive_clean["prev_lon"], drive_clean[LAT_COL], drive_clean[LON_COL]
    ) * 1000
    drive_clean.loc[drive_clean["sample_gap_seconds"].isna(), "distance_from_prev_m"] = np.nan
else:
    drive_clean["has_valid_location"] = False

clean_profile = build_data_dictionary(drive_clean, name="drive_clean_throughput_base")
clean_family_summary = summarize_column_families(clean_profile)

save_table(clean_profile, "cleaned_throughput_data_dictionary.csv")
save_table(clean_family_summary, "column_family_summary_after_cleaning.csv")

cleaning_summary = pd.DataFrame([
    {"metric": "raw_rows", "value": len(drive_raw)},
    {"metric": "raw_columns", "value": drive_raw.shape[1]},
    {"metric": "clean_rows", "value": len(drive_clean)},
    {"metric": "clean_columns", "value": drive_clean.shape[1]},
    {"metric": "columns_removed_total", "value": len(removed_columns_report)},
    {"metric": "columns_kept_total", "value": len(kept_columns_report)},
    {"metric": "missing_threshold_pct", "value": MISSING_THRESHOLD_PCT},
])
save_table(cleaning_summary, "cleaning_summary.csv")
save_parquet_or_csv(drive_clean, "drive_clean_throughput_base")

overview(drive_clean, "Cleaned throughput base")
display(cleaning_summary)

Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\cleaned_throughput_data_dictionary.csv  shape=(240, 15)
Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\column_family_summary_after_cleaning.csv  shape=(28, 8)
Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\cleaning_summary.csv  shape=(7, 2)
Parquet export failed. Falling back to compressed CSV.
Reason: ("Could not convert '8' with type str: tried to convert to double", 'Conversion failed for column Activity.Band with type object')
Saved compressed CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\drive_clean_throughput_base.csv.gz  shape=(64949, 240)
===== Cleaned throughput base =====
Shape: (64949, 240)
Rows: 64,949
Columns: 240
Memory: 226.05 MB


,column,dtype,non_null,missing,missing_pct
0,Activity.Remote IPs,object,184,64765,99.717
1,Activity.Source Ports,object,184,64765,99.717
2,Activity.TTFB,float64,184,64765,99.717
3,Activity.DNS Resolution Time,float64,186,64763,99.714
4,Activity.Remote Address,object,186,64763,99.714
...,...,...,...,...,...
235,activity_duration_seconds_from_timestamps,float64,465,64484,99.284
236,has_valid_location,bool,64949,0,0.000
237,prev_lat,float64,64403,546,0.841
238,prev_lon,float64,64403,546,0.841


,metric,value
0,raw_rows,64949.0
1,raw_columns,1365.0
2,clean_rows,64949.0
3,clean_columns,240.0
4,columns_removed_total,1145.0
5,columns_kept_total,220.0
6,missing_threshold_pct,70.0


# Canonical KPI detection

Drive-test exports contain long vendor-specific column names. This section maps important columns into short canonical names that will be used by the anomaly logic.

The original source columns remain preserved in the cleaned and modeling tables; the canonical names are only aliases for stable code and integration.

In [9]:
# =========================
# 8. Canonical throughput and LTE KPI column detection
# =========================

def find_column(
    df: pd.DataFrame,
    exact: Sequence[str] = (),
    regex: Sequence[str] = (),
    exclude_regex: Sequence[str] = (),
    prefer_most_non_null: bool = False,
) -> Optional[str]:
    """
    Find a column by exact candidates first, then regex patterns.

    exact candidates preserve the known drive-test schema.
    regex candidates make the notebook more robust if exports vary slightly.
    """
    for col in exact:
        if col in df.columns:
            return col

    candidates = []
    for col in df.columns:
        text = str(col)
        if regex and not any(re.search(p, text, flags=re.IGNORECASE) for p in regex):
            continue
        if exclude_regex and any(re.search(p, text, flags=re.IGNORECASE) for p in exclude_regex):
            continue
        candidates.append(col)

    if not candidates:
        return None
    if prefer_most_non_null:
        return max(candidates, key=lambda c: df[c].notna().sum())
    return candidates[0]


SOURCE_COLS = {
    # Target and duplicates/references.
    "target_lte_dl_tp": find_column(drive_clean, exact=[
        "LTE.LTE_Data_KPI.Agg_Throughput_DL",
        "LTE.DownlinkMeasurements.Throughput_DL",
        "LTE - PrimaryCell - PCell_Throughput",
    ], regex=[r"LTE.*(Agg_)?Throughput.*DL", r"Downlink.*Throughput"], prefer_most_non_null=True),
    "duplicate_lte_dl_tp": find_column(drive_clean, exact=["LTE.DownlinkMeasurements.Throughput_DL"]),
    "pcell_tp": find_column(drive_clean, exact=["LTE - PrimaryCell - PCell_Throughput"]),
    "scell1_tp": find_column(drive_clean, exact=["LTE - SecondaryCell 1 - SCell1_Throughput"]),
    "http_dl_tp": find_column(drive_clean, exact=["Throughput.HTTP_DL"]),
    "physical_dl_tp": find_column(drive_clean, exact=["Throughput.Physical_DL"]),
    "activity_tp": find_column(drive_clean, exact=["Activity.Throughput"]),

    # Identity and serving-cell context.
    "serving_pci": find_column(drive_clean, exact=["LTE - Serving - PCI", "LTE - PrimaryCell - PCell_PCI"]),
    "serving_earfcn": find_column(drive_clean, exact=["LTE - Serving - EARFCN_DL", "LTE - PrimaryCell - PCell_EARFCN_DL"]),
    "serving_band": find_column(drive_clean, exact=["LTE - Serving - Band", "LTE - PrimaryCell - PCell_Band"]),
    "serving_eci": find_column(drive_clean, exact=["LTE - Serving - ECI"]),
    "serving_enodebid": find_column(drive_clean, exact=["LTE - Serving - EnodeBID"]),
    "serving_lci": find_column(drive_clean, exact=["LTE - Serving - LCI"]),

    # Radio quality.
    "lte_rsrp": find_column(drive_clean, exact=["LTE - PrimaryCell Radio - RSRP"]),
    "lte_rsrq": find_column(drive_clean, exact=["LTE - PrimaryCell Radio - PCell_RSRQ"]),
    "lte_sinr": find_column(drive_clean, exact=["LTE - PrimaryCell Radio - PCell_SINR"]),
    "lte_rssi": find_column(drive_clean, exact=["LTE - PrimaryCell Radio - PCell_RSSI"]),

    # PHY/scheduler/link adaptation.
    "lte_cqi": find_column(drive_clean, exact=["LTE - PrimaryCell - PCell_CQI", "LTE - PrimaryCell - PCell_CQI_CW0"]),
    "lte_mcs": find_column(drive_clean, exact=["LTE - PrimaryCell - PCell_MCS_CW0"]),
    "lte_bler": find_column(drive_clean, exact=["LTE - PrimaryCell - PCell_BLER"]),
    "lte_rank": find_column(drive_clean, exact=["LTE - PrimaryCell - PCell_SpatialRank", "LTE - PrimaryCell - PCell_LayerNum"]),
    "lte_rb_count": find_column(drive_clean, exact=["LTE - PrimaryCell - PCell_ResourceBlockNum", "LTE.LTE_Data_KPI.Agg_RB_DL"]),
    "lte_agg_rb_dl": find_column(drive_clean, exact=["LTE.LTE_Data_KPI.Agg_RB_DL"]),
    "lte_bandwidth_dl": find_column(drive_clean, exact=["LTE - Serving - BandWidth_DL", "LTE - PrimaryCell - PCell_BandWidth_DL"]),
    "lte_agg_bandwidth_dl": find_column(drive_clean, exact=["LTE.LTE_Data_KPI.Agg_Bandwidth_DL"]),

    # Carrier aggregation / SCell1.
    "carrier_count": find_column(drive_clean, exact=["LTE.DedicatedRadioLink.CarrierCount"]),
    "scell1_rsrp": find_column(drive_clean, exact=["LTE - SecondaryCell 1 Radio - SCell1_RSRP"]),
    "scell1_rsrq": find_column(drive_clean, exact=["LTE - SecondaryCell 1 Radio - SCell1_RSRQ"]),
    "scell1_sinr": find_column(drive_clean, exact=["LTE - SecondaryCell 1 Radio - SCell1_SINR"]),
    "scell1_cqi": find_column(drive_clean, exact=["LTE - SecondaryCell 1 - SCell1_CQI", "LTE - SecondaryCell 1 - SCell1_CQI_CW0"]),
    "scell1_mcs": find_column(drive_clean, exact=["LTE - SecondaryCell 1 - SCell1_MCS_CW0"]),
    "scell1_bler": find_column(drive_clean, exact=["LTE - SecondaryCell 1 - SCell1_BLER"]),
    "scell1_rb_count": find_column(drive_clean, exact=["LTE - SecondaryCell 1 - SCell1_ResourceBlockNum"]),

    # LTE RACH direct fields, preserved as event/access evidence.
    "rach_result": find_column(drive_clean, exact=["LTE.RACH.RACH_Result"]),
    "rach_reason": find_column(drive_clean, exact=["LTE.RACH.RACH_Reason"]),
    "rach_latency": find_column(drive_clean, exact=["LTE.RACH.RACH_Latency"]),
}

canonical_mapping = pd.DataFrame([
    {"canonical_name": key, "source_column": value}
    for key, value in SOURCE_COLS.items()
])
save_table(canonical_mapping, "canonical_column_mapping.csv")
display(canonical_mapping)

if SOURCE_COLS["target_lte_dl_tp"] is None:
    raise KeyError(
        "Could not detect an LTE downlink throughput target column. "
        "Check canonical_column_mapping.csv and update SOURCE_COLS candidates."
    )

Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\canonical_column_mapping.csv  shape=(36, 2)


,canonical_name,source_column
0,target_lte_dl_tp,LTE.LTE_Data_KPI.Agg_Throughput_DL
1,duplicate_lte_dl_tp,LTE.DownlinkMeasurements.Throughput_DL
2,pcell_tp,LTE - PrimaryCell - PCell_Throughput
3,scell1_tp,LTE - SecondaryCell 1 - SCell1_Throughput
4,http_dl_tp,Throughput.HTTP_DL
5,physical_dl_tp,Throughput.Physical_DL
6,activity_tp,Activity.Throughput
7,serving_pci,LTE - Serving - PCI
8,serving_earfcn,LTE - Serving - EARFCN_DL
9,serving_band,LTE - Serving - Band


In [10]:
# =========================
# 9. Expand event and activity tables for handoff/evidence
# =========================

# Event table: one row per event. This is useful for event-window anomaly features.
events_long = build_events_long(drive_clean, id_col="id", sample_time_col="timestamp")
if not events_long.empty:
    events_long["event_family"] = events_long.apply(classify_event_family, axis=1)
    save_table(events_long, "events_long.csv")

    for col in ["Events.category", "Events.Tech", "Events.titles", "Events.subject", "event_family"]:
        if col in events_long.columns:
            summary = events_long[col].fillna("Unknown").astype(str).value_counts().reset_index()
            summary.columns = [col, "event_count"]
            save_table(summary, f"event_summary_{col.replace('.', '_').replace(' ', '_')}.csv")
else:
    print("No events were available to expand.")

# Activity table: preserve sparse activity records for the later RCA colleague.
activity_cols = [c for c in drive_clean.columns if str(c).startswith("Activity.")]
activity_key_cols = [c for c in ["id", "timestamp", "id_route_test_label", "id_test_type"] if c in drive_clean.columns]

if activity_cols:
    activity_table = drive_clean[activity_key_cols + activity_cols].copy()
    activity_table = activity_table[activity_table[activity_cols].notna().any(axis=1)].reset_index(drop=True)
    save_table(activity_table, "activity_table.csv")
    if "Activity.Activity" in activity_table.columns:
        activity_summary = activity_table["Activity.Activity"].fillna("Unknown").astype(str).value_counts().reset_index()
        activity_summary.columns = ["activity", "rows"]
        save_table(activity_summary, "activity_type_summary.csv")
else:
    activity_table = pd.DataFrame()
    print("No Activity.* columns found.")

print("Events long shape:", events_long.shape)
print("Activity table shape:", activity_table.shape)

Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\events_long.csv  shape=(102302, 13)
Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\event_summary_Events_category.csv  shape=(4, 2)
Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\event_summary_Events_Tech.csv  shape=(3, 2)
Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\event_summary_Events_titles.csv  shape=(117, 2)
Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\event_summary_Events_subject.csv  shape=(15, 2)
Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\event_summary_event_family.csv  shape=(8, 2)
Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\activity_table.csv  shape=(465, 37)
Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\activity_type_summary.csv  shape=(2, 2)
Events long shape: (102302, 13)
Activity table shape: (465, 37)

# Build the LTE-DL throughput modeling table

This is the row-level table used by the statistical anomaly detectors.

V1 filter logic:

1. LTE measurement exists.
2. LTE downlink throughput target exists and is non-negative.
3. If `id_test_type` exists, keep download rows containing `DL`.

MO-call and WCDMA-heavy rows are preserved in the cleaned base, but they are not mixed into the first LTE throughput anomaly table.

In [11]:
# =========================
# 10. Canonical aliases and LTE-DL filtering
# =========================

lte = drive_clean.copy()

# Canonical metadata and location aliases.
lte["latitude"] = to_numeric_series(lte, LAT_COL)
lte["longitude"] = to_numeric_series(lte, LON_COL)
lte["speed"] = to_numeric_series(lte, SPEED_COL)
lte["location_accuracy"] = to_numeric_series(lte, ACCURACY_COL)
lte["altitude"] = to_numeric_series(lte, ALTITUDE_COL)

# Canonical LTE target and context aliases.
lte["actual_lte_dl_throughput"] = to_numeric_series(lte, SOURCE_COLS["target_lte_dl_tp"]) * THROUGHPUT_TO_MBPS
lte["serving_pci"] = to_numeric_series(lte, SOURCE_COLS["serving_pci"])
lte["serving_earfcn"] = to_numeric_series(lte, SOURCE_COLS["serving_earfcn"])
lte["serving_band"] = to_numeric_series(lte, SOURCE_COLS["serving_band"])
lte["serving_eci"] = to_numeric_series(lte, SOURCE_COLS["serving_eci"])
lte["serving_enodebid"] = to_numeric_series(lte, SOURCE_COLS["serving_enodebid"])
lte["serving_lci"] = to_numeric_series(lte, SOURCE_COLS["serving_lci"])

# Radio/PHY/CA aliases.
for alias in [
    "lte_rsrp", "lte_rsrq", "lte_sinr", "lte_rssi", "lte_cqi", "lte_mcs", "lte_bler",
    "lte_rank", "lte_rb_count", "lte_agg_rb_dl", "lte_bandwidth_dl", "lte_agg_bandwidth_dl",
    "carrier_count", "scell1_rsrp", "scell1_rsrq", "scell1_sinr", "scell1_cqi",
    "scell1_mcs", "scell1_bler", "scell1_rb_count", "rach_latency",
]:
    lte[alias] = to_numeric_series(lte, SOURCE_COLS.get(alias))

# Categorical direct aliases.
for alias in ["rach_result", "rach_reason"]:
    src = SOURCE_COLS.get(alias)
    lte[alias] = lte[src] if src in lte.columns else np.nan

# Carrier count fallback: if missing, infer 2 carriers when SCell1 metrics exist, otherwise 1.
scell_present = lte[[c for c in ["scell1_rsrp", "scell1_sinr", "scell1_cqi", "scell1_rb_count"] if c in lte.columns]].notna().any(axis=1)
carrier_count_fallback = pd.Series(np.where(scell_present, 2, 1), index=lte.index, dtype="float64")
lte["carrier_count"] = lte["carrier_count"].fillna(carrier_count_fallback)

# Availability flags.
lte["has_lte_measurement"] = lte[["lte_rsrp", "lte_rsrq", "lte_sinr", "serving_pci", "serving_earfcn"]].notna().any(axis=1)
lte["has_lte_dl_throughput"] = lte["actual_lte_dl_throughput"].notna() & (lte["actual_lte_dl_throughput"] >= 0)

# Download-test filter from parsed id_test_type when available.
if "id_test_type" in lte.columns:
    lte["is_download_test"] = lte["id_test_type"].astype(str).str.contains("DL", case=False, na=False)
else:
    lte["is_download_test"] = True

filter_mask = lte["has_lte_measurement"] & lte["has_lte_dl_throughput"] & lte["is_download_test"]
lte_dl = lte.loc[filter_mask].copy().reset_index(drop=False).rename(columns={"index": "source_index"})

# Fallback warning: if the DL filter gives zero rows, use LTE throughput rows and let the analyst inspect id_test_type.
if lte_dl.empty:
    print("Warning: LTE-DL filter returned zero rows. Falling back to all LTE rows with valid DL throughput.")
    filter_mask = lte["has_lte_measurement"] & lte["has_lte_dl_throughput"]
    lte_dl = lte.loc[filter_mask].copy().reset_index(drop=False).rename(columns={"index": "source_index"})

print("LTE-DL modeling table shape:", lte_dl.shape)
print("Throughput target source column:", SOURCE_COLS["target_lte_dl_tp"])
print("Throughput stats in Mbps:")
display(lte_dl["actual_lte_dl_throughput"].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99]).to_frame())


# -------------------------
# Download activity-window context
# -------------------------

def add_download_activity_window_flags(
    samples: pd.DataFrame,
    activities: pd.DataFrame,
    session_col: str = "id",
    sample_time_col: str = "timestamp",
    activity_name_col: str = "Activity.Activity",
    activity_start_col: str = "Activity.Start TimeStamp_parsed",
    activity_end_col: str = "Activity.End TimeStamp_parsed",
    activity_status_col: str = "Activity.Status",
    activity_pattern: str = DOWNLOAD_ACTIVITY_NAME_PATTERN,
    warmup_seconds: int = DOWNLOAD_ACTIVITY_WARMUP_SECONDS,
    cooldown_seconds: int = DOWNLOAD_ACTIVITY_COOLDOWN_SECONDS,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Mark samples that fall inside HTTP/HTTPS download activity windows.

    Why this is needed:
    LTE-DL session rows may include script/activity timing such as HTTP transfer start/end.
    In earlier versions this was used as a hard anomaly-analysis mask. In the latest version
    it is retained only as context because the active/warmup/outside throughput distributions
    were close enough to analyze all valid LTE-DL rows together. RB usage is now the primary
    demand/resource-confidence signal.

    Returns:
    - samples with raw and analysis-window flags.
    - activity intervals used for filtering.
    """
    out = samples.copy()
    out["download_activity_window_raw_flag"] = False
    out["download_analysis_window_flag"] = True
    out["download_context_label"] = "activity_filter_not_available"

    if activities is None or activities.empty:
        return out, pd.DataFrame()

    required = {session_col, activity_name_col, activity_start_col, activity_end_col}
    if not required.issubset(activities.columns) or sample_time_col not in out.columns:
        return out, pd.DataFrame()

    acts = activities.copy()
    acts = acts[acts[activity_name_col].astype(str).str.contains(activity_pattern, case=False, na=False)].copy()
    if acts.empty:
        return out, pd.DataFrame()

    acts[activity_start_col] = pd.to_datetime(acts[activity_start_col], utc=True, errors="coerce")
    acts[activity_end_col] = pd.to_datetime(acts[activity_end_col], utc=True, errors="coerce")
    acts = acts.dropna(subset=[session_col, activity_start_col, activity_end_col]).copy()
    acts = acts[acts[activity_end_col] > acts[activity_start_col]].copy()
    if acts.empty:
        return out, pd.DataFrame()

    warmup = pd.to_timedelta(warmup_seconds, unit="s")
    cooldown = pd.to_timedelta(cooldown_seconds, unit="s")
    acts["download_raw_start_utc"] = acts[activity_start_col]
    acts["download_raw_end_utc"] = acts[activity_end_col]
    acts["download_analysis_start_utc"] = acts[activity_start_col] + warmup
    acts["download_analysis_end_utc"] = acts[activity_end_col] - cooldown

    # If the activity is too short for the warmup/cooldown margin, fall back to the raw window.
    invalid_analysis = acts["download_analysis_end_utc"] <= acts["download_analysis_start_utc"]
    acts.loc[invalid_analysis, "download_analysis_start_utc"] = acts.loc[invalid_analysis, "download_raw_start_utc"]
    acts.loc[invalid_analysis, "download_analysis_end_utc"] = acts.loc[invalid_analysis, "download_raw_end_utc"]

    out["_sample_timestamp_utc"] = pd.to_datetime(out[sample_time_col], utc=True, errors="coerce")
    out["download_analysis_window_flag"] = False
    out["download_context_label"] = "outside_download_activity"

    for sid, g in acts.groupby(session_col, sort=False):
        sample_idx = out.index[out[session_col].eq(sid) & out["_sample_timestamp_utc"].notna()]
        if len(sample_idx) == 0:
            continue

        sample_times = out.loc[sample_idx, "_sample_timestamp_utc"]
        raw_mask = pd.Series(False, index=sample_idx)
        analysis_mask = pd.Series(False, index=sample_idx)

        for _, act in g.iterrows():
            raw_mask |= sample_times.between(act["download_raw_start_utc"], act["download_raw_end_utc"], inclusive="both")
            analysis_mask |= sample_times.between(act["download_analysis_start_utc"], act["download_analysis_end_utc"], inclusive="both")

        out.loc[sample_idx, "download_activity_window_raw_flag"] |= raw_mask.values
        out.loc[sample_idx, "download_analysis_window_flag"] |= analysis_mask.values

    # Context labels.
    out.loc[out["download_activity_window_raw_flag"] & ~out["download_analysis_window_flag"], "download_context_label"] = "download_warmup_or_cooldown"
    out.loc[out["download_analysis_window_flag"], "download_context_label"] = "active_download_analysis_window"

    out = out.drop(columns=["_sample_timestamp_utc"])
    return out, acts.reset_index(drop=True)


if BUILD_DOWNLOAD_ACTIVITY_CONTEXT_FLAGS and "activity_table" in globals():
    lte_dl, download_activity_intervals = add_download_activity_window_flags(lte_dl, activity_table)
else:
    lte_dl["download_activity_window_raw_flag"] = False
    lte_dl["download_analysis_window_flag"] = True
    lte_dl["download_context_label"] = "activity_context_flags_disabled"
    download_activity_intervals = pd.DataFrame()

# If no valid download intervals are found, keep all LTE-DL rows analyzable rather than silently removing everything.
if BUILD_DOWNLOAD_ACTIVITY_CONTEXT_FLAGS and not bool(lte_dl["download_analysis_window_flag"].any()):
    print("Warning: no active HTTP/HTTPS download windows were detected. Falling back to analyzing all LTE-DL rows.")
    lte_dl["download_analysis_window_flag"] = True
    lte_dl["download_context_label"] = "activity_context_fallback_all_rows"

lte_dl["inactive_download_context_flag"] = ~lte_dl["download_analysis_window_flag"].fillna(True).astype(bool)

download_window_summary = (
    lte_dl["download_context_label"]
    .fillna("Unknown")
    .value_counts(dropna=False)
    .rename_axis("download_context_label")
    .reset_index(name="rows")
)
download_window_summary["pct_of_lte_dl_rows"] = (download_window_summary["rows"] / max(len(lte_dl), 1) * 100).round(3)
save_table(download_window_summary, "download_activity_window_summary.csv")
display(download_window_summary)

if not download_activity_intervals.empty:
    interval_summary = download_activity_intervals.copy()
    interval_summary["raw_duration_seconds"] = (
        interval_summary["download_raw_end_utc"] - interval_summary["download_raw_start_utc"]
    ).dt.total_seconds()
    interval_summary["analysis_duration_seconds"] = (
        interval_summary["download_analysis_end_utc"] - interval_summary["download_analysis_start_utc"]
    ).dt.total_seconds()
    keep_cols = [
        c for c in [
            "id", "Activity.Activity", "Activity.Status", "Activity.Tech",
            "download_raw_start_utc", "download_raw_end_utc",
            "download_analysis_start_utc", "download_analysis_end_utc",
            "raw_duration_seconds", "analysis_duration_seconds",
        ]
        if c in interval_summary.columns
    ]
    save_table(interval_summary[keep_cols], "download_activity_intervals_used.csv")


LTE-DL modeling table shape: (32238, 279)
Throughput target source column: LTE.LTE_Data_KPI.Agg_Throughput_DL
Throughput stats in Mbps:


,actual_lte_dl_throughput
count,32238.000000
mean,50.859819
std,45.106051
min,0.000056
1%,0.016027
5%,1.321483
10%,4.990454
25%,15.431080
50%,38.801480
75%,74.473165


Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\download_activity_window_summary.csv  shape=(3, 3)


,download_context_label,rows,pct_of_lte_dl_rows
0,outside_download_activity,20475,63.512
1,active_download_analysis_window,11372,35.275
2,download_warmup_or_cooldown,391,1.213


Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\download_activity_intervals_used.csv  shape=(191, 10)


In [12]:
# =========================
# 11. Quality bins, flags, neighbor aggregates, and event-window features
# =========================

def bin_rsrp(x):
    """Bin LTE RSRP using the calibrated drive-test thresholds."""
    if pd.isna(x): return np.nan
    if x >= RSRP_EXCELLENT_MIN_DBM: return "Excellent"
    if x >= RSRP_GOOD_MIN_DBM: return "Good"
    if x >= RSRP_FAIR_MIN_DBM: return "Fair"
    return "Poor"


def bin_rsrq(x):
    """Bin LTE RSRQ using the calibrated drive-test thresholds."""
    if pd.isna(x): return np.nan
    if x >= RSRQ_EXCELLENT_MIN_DB: return "Excellent"
    if x >= RSRQ_GOOD_MIN_DB: return "Good"
    if x >= RSRQ_FAIR_MIN_DB: return "Fair"
    return "Poor"


def bin_sinr(x):
    """Bin LTE SINR using the calibrated drive-test thresholds."""
    if pd.isna(x): return np.nan
    if x >= SINR_EXCELLENT_MIN_DB: return "Excellent"
    if x >= SINR_GOOD_MIN_DB: return "Good"
    if x >= SINR_FAIR_MIN_DB: return "Fair"
    return "Poor"


# Quality bins.
lte_dl["rsrp_bin"] = lte_dl["lte_rsrp"].map(bin_rsrp)
lte_dl["rsrq_bin"] = lte_dl["lte_rsrq"].map(bin_rsrq)
lte_dl["sinr_bin"] = lte_dl["lte_sinr"].map(bin_sinr)

radio_bin_summary = pd.concat(
    {
        "RSRP": lte_dl["rsrp_bin"].value_counts(dropna=False, normalize=True).mul(100).round(2),
        "RSRQ": lte_dl["rsrq_bin"].value_counts(dropna=False, normalize=True).mul(100).round(2),
        "SINR": lte_dl["sinr_bin"].value_counts(dropna=False, normalize=True).mul(100).round(2),
    },
    axis=1,
).fillna(0).sort_index()
print("Radio-quality bin distribution (% of LTE-DL samples):")
display(radio_bin_summary)

# Evidence flags used by the anomaly labels. These are not final RCA causes.
# Poor flags use the lower edge of the Fair bin.
# This means Fair radio is preserved as warning/context, while Poor radio is reserved
# for clearly bad field conditions.
lte_dl["poor_rsrp_flag"] = lte_dl["lte_rsrp"] < RSRP_FAIR_MIN_DBM
lte_dl["poor_rsrq_flag"] = lte_dl["lte_rsrq"] < RSRQ_FAIR_MIN_DB
lte_dl["poor_sinr_flag"] = lte_dl["lte_sinr"] < SINR_FAIR_MIN_DB
lte_dl["high_bler_flag"] = lte_dl["lte_bler"] > 10
lte_dl["low_cqi_flag"] = lte_dl["lte_cqi"] < 7

# Low RB allocation is relative because RB values depend on bandwidth and export semantics.
rb_q25 = lte_dl["lte_rb_count"].quantile(0.25) if lte_dl["lte_rb_count"].notna().any() else np.nan
lte_dl["low_rb_allocation_flag"] = lte_dl["lte_rb_count"].notna() & (lte_dl["lte_rb_count"] <= rb_q25)

# Neighbor aggregates from wide LTE NeighborCells(n) columns.
def add_lte_neighbor_aggregates(df_samples: pd.DataFrame) -> pd.DataFrame:
    out = df_samples.copy()
    rsrp_cols = [c for c in out.columns if re.match(r"LTE - NeighborCells\(\d+\) - RSRP$", str(c))]
    pci_cols = [c for c in out.columns if re.match(r"LTE - NeighborCells\(\d+\) - PCI$", str(c))]
    if rsrp_cols:
        rsrp_numeric = out[rsrp_cols].apply(pd.to_numeric, errors="coerce")
        out["neighbor_count"] = rsrp_numeric.notna().sum(axis=1)
        out["best_neighbor_rsrp"] = rsrp_numeric.max(axis=1)
        out["strong_neighbor_count"] = (rsrp_numeric >= -100).sum(axis=1)
        out["serving_minus_best_neighbor_rsrp"] = out["lte_rsrp"] - out["best_neighbor_rsrp"]
    else:
        out["neighbor_count"] = 0
        out["best_neighbor_rsrp"] = np.nan
        out["strong_neighbor_count"] = 0
        out["serving_minus_best_neighbor_rsrp"] = np.nan

    # PCI count is useful for overlap/mobility context.
    if pci_cols:
        out["neighbor_pci_count"] = out[pci_cols].notna().sum(axis=1)
    else:
        out["neighbor_pci_count"] = 0
    return out


lte_dl = add_lte_neighbor_aggregates(lte_dl)

# Event-window features.
# IMPORTANT CAUSALITY RULE:
# - *_count_prevXs / near_*_prev_flag use ONLY events at or before the sample timestamp and
#   are safe candidates for causal forecasting.
# - legacy *_count_pmXs / near_*_flag are symmetric ±X-second OFFLINE RCA CONTEXT ONLY.
#   Notebook 02 explicitly excludes them from causal model predictors.
def add_event_window_features(samples: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    """Add both causal past-only and symmetric offline event context efficiently."""
    out = samples.copy()

    def _init_empty(frame: pd.DataFrame) -> pd.DataFrame:
        for name, window in EVENT_WINDOWS_SECONDS.items():
            frame[f"{name}_count_prev{window}s"] = 0
            frame[f"near_{name}_prev_flag"] = False
            # Legacy/offline context only.
            frame[f"{name}_count_pm{window}s"] = 0
            frame[f"near_{name}_flag"] = False
        return frame

    if events.empty or "event_timestamp" not in events.columns or "timestamp" not in out.columns:
        return _init_empty(out)

    ev = events.copy()
    ev["event_timestamp_utc"] = pd.to_datetime(ev["event_timestamp"], errors="coerce", utc=True)
    out["timestamp_utc_for_event_window"] = pd.to_datetime(out["timestamp"], errors="coerce", utc=True)
    ev = ev.dropna(subset=["event_timestamp_utc"])
    if ev.empty:
        return _init_empty(out).drop(columns=["timestamp_utc_for_event_window"], errors="ignore")

    id_col = "id" if "id" in out.columns and "id" in ev.columns else None
    ev["event_time_ns"] = ev["event_timestamp_utc"].astype("int64")

    any_event_times, family_event_times = {}, {}
    if id_col:
        for sid, group in ev.groupby(id_col, dropna=False):
            any_event_times[sid] = np.sort(group["event_time_ns"].values)
        for (sid, family), group in ev.groupby([id_col, "event_family"], dropna=False):
            family_event_times[(sid, family)] = np.sort(group["event_time_ns"].values)
    else:
        any_event_times[None] = np.sort(ev["event_time_ns"].values)
        for family, group in ev.groupby("event_family", dropna=False):
            family_event_times[(None, family)] = np.sort(group["event_time_ns"].values)

    out = _init_empty(out)
    sample_groups = out.groupby(id_col, dropna=False) if id_col else [(None, out)]

    for sid, sample_group in sample_groups:
        sample_idx = sample_group.index
        sample_times = out.loc[sample_idx, "timestamp_utc_for_event_window"].astype("int64").values

        for family_name, window_seconds in EVENT_WINDOWS_SECONDS.items():
            if family_name == "any_event":
                event_times = any_event_times.get(sid if id_col else None)
            else:
                event_times = family_event_times.get((sid if id_col else None, family_name))
            if event_times is None or len(event_times) == 0:
                continue

            radius_ns = int(window_seconds * 1e9)

            # Causal window [t-window, t]. No future event is visible.
            causal_left = np.searchsorted(event_times, sample_times - radius_ns, side="left")
            causal_right = np.searchsorted(event_times, sample_times, side="right")
            causal_counts = causal_right - causal_left
            out.loc[sample_idx, f"{family_name}_count_prev{window_seconds}s"] = causal_counts
            out.loc[sample_idx, f"near_{family_name}_prev_flag"] = causal_counts > 0

            # Symmetric offline context [t-window, t+window]. NEVER use for causal forecasting.
            sym_left = np.searchsorted(event_times, sample_times - radius_ns, side="left")
            sym_right = np.searchsorted(event_times, sample_times + radius_ns, side="right")
            sym_counts = sym_right - sym_left
            out.loc[sample_idx, f"{family_name}_count_pm{window_seconds}s"] = sym_counts
            out.loc[sample_idx, f"near_{family_name}_flag"] = sym_counts > 0

    return out.drop(columns=["timestamp_utc_for_event_window"], errors="ignore")


lte_dl = add_event_window_features(lte_dl, events_long)

# Legacy compatibility aliases after the refined event-family split.
# The old broad handover_mobility flag now means actual HO execution or HO failure,
# not ordinary A1/A2/A3/A5/A6 measurement reports.
def _sum_existing_event_counts(df: pd.DataFrame, names: Sequence[str], window: int) -> pd.Series:
    cols = [f"{name}_count_pm{window}s" for name in names if f"{name}_count_pm{window}s" in df.columns]
    if not cols:
        return pd.Series(0, index=df.index)
    return df[cols].fillna(0).sum(axis=1)

lte_dl["handover_mobility_count_pm5s"] = _sum_existing_event_counts(lte_dl, ["handover_execution", "handover_failure"], 5)
lte_dl["near_handover_mobility_flag"] = lte_dl["handover_mobility_count_pm5s"] > 0
lte_dl["ca_change_count_pm10s"] = lte_dl.get("ca_activation_change_count_pm10s", pd.Series(0, index=lte_dl.index)).fillna(0)
lte_dl["near_ca_change_flag"] = lte_dl["ca_change_count_pm10s"] > 0
lte_dl["rach_count_pm10s"] = _sum_existing_event_counts(lte_dl, ["rach_attempt", "rach_failure"], 10)
lte_dl["near_rach_flag"] = lte_dl["rach_count_pm10s"] > 0

# Direct RACH result flag if row-level RACH fields exist.
lte_dl["rach_failure_direct_flag"] = lte_dl["rach_result"].astype(str).str.contains("fail|timeout|reject", case=False, na=False)

print("Feature-enriched LTE-DL table shape:", lte_dl.shape)
display(lte_dl[[
    "actual_lte_dl_throughput", "lte_rsrp", "lte_rsrq", "lte_sinr", "lte_cqi", "lte_bler",
    "carrier_count", "neighbor_count", "best_neighbor_rsrp", "near_any_event_flag",
    "near_handover_mobility_flag", "near_rach_failure_flag", "near_ca_change_flag"
]].head(20))

# No generic causal event aliases are created here. The ML notebook consumes the explicit
# past-only event families directly (for example handover_execution_count_prev5s and
# handover_failure_count_prev10s), which avoids mixing differently sized causal windows.

causal_event_feature_cols = sorted([
    c for c in lte_dl.columns if ("_count_prev" in c or c.endswith("_prev_flag"))
])
symmetric_context_feature_cols = sorted([
    c for c in lte_dl.columns if ("_count_pm" in c or (c.startswith("near_") and not c.endswith("_prev_flag")))
])
causality_audit = pd.DataFrame([
    {"column": c, "feature_role": "CAUSAL_PAST_ONLY_MODEL_ELIGIBLE"} for c in causal_event_feature_cols
] + [
    {"column": c, "feature_role": "SYMMETRIC_OFFLINE_RCA_CONTEXT_ONLY"} for c in symmetric_context_feature_cols
])
save_table(causality_audit, "event_feature_causality_audit.csv", category="preprocessing")
print("Causal event features:", len(causal_event_feature_cols))
print("Symmetric offline-only event context features:", len(symmetric_context_feature_cols))


Radio-quality bin distribution (% of LTE-DL samples):


,RSRP,RSRQ,SINR
Excellent,31.61,45.65,36.87
Fair,35.16,3.77,36.22
Good,32.58,50.37,25.46
Poor,0.65,0.21,1.45


Feature-enriched LTE-DL table shape: (32238, 336)


,actual_lte_dl_throughput,lte_rsrp,lte_rsrq,lte_sinr,lte_cqi,lte_bler,carrier_count,neighbor_count,best_neighbor_rsrp,near_any_event_flag,near_handover_mobility_flag,near_rach_failure_flag,near_ca_change_flag
0,0.000144,-101.905240,-8.467742,10.830000,NaN,0.000000,1.0,3,-116.156250,True,False,False,True
1,0.002336,-105.852940,-9.630515,7.888544,8.0,0.000000,1.0,3,-112.031250,True,False,False,True
2,0.031944,-104.577126,-9.476728,8.440907,8.0,11.764706,1.0,3,-114.369790,True,False,False,True
3,0.043088,-100.460590,-9.152027,9.509525,9.0,9.090909,1.0,3,-106.708336,True,False,False,True
4,0.001464,-107.604164,-9.046296,5.791666,10.0,0.000000,1.0,4,-104.718750,True,False,False,True
5,0.008488,-110.119080,-10.038816,4.092308,8.0,0.000000,2.0,4,-107.906250,True,False,False,True
6,0.018432,-111.684784,-10.729348,1.961682,6.0,4.166666,2.0,4,-114.062500,True,False,False,True
7,0.008776,-111.661766,-10.604412,2.067532,8.0,0.000000,1.0,2,-115.582030,True,False,False,True
8,0.008024,-111.179110,-10.731343,2.957142,8.0,0.000000,1.0,2,-115.000000,True,False,False,True
9,0.019640,-110.680310,-10.403735,3.066666,8.0,0.000000,1.0,2,-111.375000,True,False,False,True


Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\event_feature_causality_audit.csv  shape=(38, 2)
Causal event features: 16
Symmetric offline-only event context features: 22


# Visual EDA before anomaly detection

These plots are placed before the anomaly rules on purpose. They answer: **what does the LTE-DL data look like before we start flagging anything?**

This helps validate whether thresholds such as 10 Mbps, RSRP/RSRQ/SINR bins, and carrier-count expectations are reasonable for this real drive-test export.

In [13]:
# =========================
# 11B. Pre-anomaly visual EDA and threshold sanity checks
# =========================

if not lte_dl.empty:
    tp_quantiles = lte_dl["actual_lte_dl_throughput"].quantile([0.01, 0.05, 0.10, 0.17, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).rename("throughput_mbps")
    threshold_sanity = pd.DataFrame({
        "metric": [
            "rows",
            "throughput_p01", "throughput_p05", "throughput_p10", "throughput_p17", "throughput_p25", "throughput_median", "throughput_p75", "throughput_p90", "throughput_p95", "throughput_p99",
            "pct_below_5_mbps", "pct_below_10_mbps", "pct_below_15_mbps", "pct_below_20_mbps",
        ],
        "value": [
            len(lte_dl),
            tp_quantiles.loc[0.01], tp_quantiles.loc[0.05], tp_quantiles.loc[0.10], tp_quantiles.loc[0.17], tp_quantiles.loc[0.25], tp_quantiles.loc[0.50], tp_quantiles.loc[0.75], tp_quantiles.loc[0.90], tp_quantiles.loc[0.95], tp_quantiles.loc[0.99],
            (lte_dl["actual_lte_dl_throughput"] < 5).mean() * 100,
            (lte_dl["actual_lte_dl_throughput"] < 10).mean() * 100,
            (lte_dl["actual_lte_dl_throughput"] < 15).mean() * 100,
            (lte_dl["actual_lte_dl_throughput"] < 20).mean() * 100,
        ],
    })
    threshold_sanity["value"] = threshold_sanity["value"].astype(float).round(3)
    save_table(threshold_sanity, "throughput_threshold_sanity.csv")
    display(threshold_sanity)

    # Time-spacing diagnostic: rolling remains sample-based, but this proves whether
    # sample rows are close to consecutive seconds.
    if "sample_gap_seconds" in lte_dl.columns:
        plot_distribution_with_lines(
            lte_dl,
            "sample_gap_seconds",
            "Sample Gap Distribution for LTE-DL Rows",
            "Seconds since previous sample in same session",
            thresholds={"1 second": 1, "2 seconds": 2},
            bins=40,
            filename="02b_sample_gap_seconds_distribution.png",
            clip_upper_q=0.99,
        )

    # Compare throughput inside and outside the active download analysis window.
    # This context is retained mainly as confidence/evidence. If the medians are similar,
    # do not use separate thresholds for each download-context bucket.
    if "download_context_label" in lte_dl.columns:
        plot_box_by_category(
            lte_dl,
            "download_context_label",
            "actual_lte_dl_throughput",
            "Throughput by Download Activity Context",
            "LTE DL throughput (Mbps)",
            filename="02c_throughput_by_download_activity_context.png",
            category_order=[
                "active_download_analysis_window",
                "download_warmup_or_cooldown",
                "outside_download_activity",
                "activity_filter_not_available",
                "activity_filter_disabled",
                "activity_filter_fallback_all_rows",
            ],
        )

        download_context_summary = (
            lte_dl.groupby("download_context_label", dropna=False)["actual_lte_dl_throughput"]
            .agg(
                rows="count",
                mean="mean",
                p10=lambda s: s.quantile(0.10),
                p25=lambda s: s.quantile(0.25),
                median="median",
                p75=lambda s: s.quantile(0.75),
                pct_below_10=lambda s: (s < LOW_TP_FIXED_MBPS).mean() * 100,
            )
            .reset_index()
        )
        for c in ["mean", "p10", "p25", "median", "p75", "pct_below_10"]:
            download_context_summary[c] = download_context_summary[c].astype(float).round(3)
        save_table(download_context_summary, "throughput_by_download_context_summary.csv")
        display(download_context_summary)

    plot_distribution_with_lines(
        lte_dl,
        "actual_lte_dl_throughput",
        "LTE-DL Throughput Distribution Before Anomaly Detection",
        "LTE DL throughput (Mbps)",
        thresholds={
            "Very low threshold": VERY_LOW_TP_MBPS,
            "Low threshold": LOW_TP_FIXED_MBPS,
            "Global P10": float(tp_quantiles.loc[0.10]),
            "Global P25": float(tp_quantiles.loc[0.25]),
            "Median": float(tp_quantiles.loc[0.50]),
        },
        filename="01_pre_anomaly_throughput_distribution.png",
        clip_upper_q=HISTOGRAM_COLOR_CLIP_Q,
    )

    radio_order = ["Poor", "Fair", "Good", "Excellent"]
    plot_box_by_category(lte_dl, "rsrp_bin", "actual_lte_dl_throughput", "Throughput by RSRP Bin", "LTE DL throughput (Mbps)", "02_throughput_by_rsrp_bin.png", radio_order)
    plot_box_by_category(lte_dl, "rsrq_bin", "actual_lte_dl_throughput", "Throughput by RSRQ Bin", "LTE DL throughput (Mbps)", "03_throughput_by_rsrq_bin.png", radio_order)
    plot_box_by_category(lte_dl, "sinr_bin", "actual_lte_dl_throughput", "Throughput by SINR Bin", "LTE DL throughput (Mbps)", "04_throughput_by_sinr_bin.png", radio_order)

    # Carrier aggregation / CA context.
    lte_dl["ca_condition_label"] = np.select(
        [
            lte_dl["carrier_count"].fillna(0) >= 3,
            lte_dl["carrier_count"].fillna(0) == 2,
            lte_dl["carrier_count"].fillna(0) == 1,
        ],
        ["3CC_or_more_CA", "2CC_CA", "single_carrier"],
        default="unknown_carrier_count",
    )
    plot_box_by_category(lte_dl, "carrier_count", "actual_lte_dl_throughput", "Throughput by LTE Carrier Count", "LTE DL throughput (Mbps)", "05_throughput_by_carrier_count.png")
    plot_box_by_category(lte_dl, "ca_condition_label", "actual_lte_dl_throughput", "Throughput by CA Condition", "LTE DL throughput (Mbps)", "05b_throughput_by_ca_condition.png", category_order=["single_carrier", "2CC_CA", "3CC_or_more_CA", "unknown_carrier_count"])

    ca_summary = (
        lte_dl.groupby(["ca_condition_label", "carrier_count"], dropna=False)["actual_lte_dl_throughput"]
        .agg(
            rows="count",
            mean="mean",
            p10=lambda s: s.quantile(0.10),
            p25=lambda s: s.quantile(0.25),
            median="median",
            p75=lambda s: s.quantile(0.75),
            p90=lambda s: s.quantile(0.90),
            pct_below_10=lambda s: (s < LOW_TP_FIXED_MBPS).mean() * 100,
        )
        .reset_index()
        .sort_values(["carrier_count", "ca_condition_label"])
    )
    for c in ["mean", "p10", "p25", "median", "p75", "p90", "pct_below_10"]:
        ca_summary[c] = ca_summary[c].astype(float).round(3)
    save_table(ca_summary, "throughput_by_ca_condition_summary.csv")
    display(ca_summary)

    # RB usage reliability check. This tells us whether RB allocation can be trusted as evidence.
    rb_cols = [c for c in ["lte_rb_count", "lte_agg_rb_dl", "scell1_rb_count"] if c in lte_dl.columns]
    rb_quality_rows = []
    for col in rb_cols:
        s = pd.to_numeric(lte_dl[col], errors="coerce")
        non_null_pct = s.notna().mean() * 100
        unique_values = int(s.nunique(dropna=True))
        reliable = (non_null_pct >= MIN_RELIABLE_RB_NON_NULL_PCT) and (unique_values >= MIN_RELIABLE_RB_UNIQUE_VALUES)
        rb_quality_rows.append({
            "rb_column": col,
            "non_null_rows": int(s.notna().sum()),
            "non_null_pct": round(non_null_pct, 3),
            "unique_values": unique_values,
            "min": round(float(s.min()), 3) if s.notna().any() else np.nan,
            "p10": round(float(s.quantile(0.10)), 3) if s.notna().any() else np.nan,
            "p25": round(float(s.quantile(0.25)), 3) if s.notna().any() else np.nan,
            "median": round(float(s.median()), 3) if s.notna().any() else np.nan,
            "p75": round(float(s.quantile(0.75)), 3) if s.notna().any() else np.nan,
            "p90": round(float(s.quantile(0.90)), 3) if s.notna().any() else np.nan,
            "max": round(float(s.max()), 3) if s.notna().any() else np.nan,
            "reliable_for_anomaly_evidence": reliable,
        })
    rb_usage_quality_summary = pd.DataFrame(rb_quality_rows)
    save_table(rb_usage_quality_summary, "rb_usage_quality_summary.csv")
    display(rb_usage_quality_summary)

    # Make a global flag so later anomaly logic can decide whether low-RB evidence should be trusted.
    reliable_rb_columns = rb_usage_quality_summary.loc[rb_usage_quality_summary["reliable_for_anomaly_evidence"], "rb_column"].tolist() if not rb_usage_quality_summary.empty else []
    lte_dl["rb_usage_evidence_reliable_flag"] = bool(reliable_rb_columns)
    lte_dl["reliable_rb_columns"] = ";".join(reliable_rb_columns) if reliable_rb_columns else ""


    # Demand/RB interpretation.
    # This is very important for throughput anomaly detection: low observed throughput is
    # only a strong capacity/efficiency anomaly when there is evidence that the scheduler
    # allocated enough resources. RB usage is therefore the main demand/resource-confidence
    # signal. Activity-window labels remain context only.
    preferred_rb_order = [c for c in ["lte_agg_rb_dl", "lte_rb_count", "scell1_rb_count"] if c in reliable_rb_columns]
    rb_reference_col = preferred_rb_order[0] if preferred_rb_order else None
    lte_dl["rb_reference_column"] = rb_reference_col if rb_reference_col else "none_reliable"

    if rb_reference_col:
        rb_values = pd.to_numeric(lte_dl[rb_reference_col], errors="coerce")
        # Use all valid LTE-DL rows for RB quantiles. Activity-window labels are context only.
        rb_reference_mask = rb_values.notna()
        rb_reference_values = rb_values.where(rb_reference_mask)

        rb_low_threshold = float(rb_reference_values.quantile(LOW_RB_QUANTILE))
        rb_medium_threshold = float(rb_reference_values.quantile(MEDIUM_RB_QUANTILE))
        rb_high_threshold = float(rb_reference_values.quantile(HIGH_RB_QUANTILE))

        lte_dl["rb_usage_value"] = rb_values
        lte_dl["rb_low_usage_threshold"] = rb_low_threshold
        lte_dl["rb_medium_usage_threshold"] = rb_medium_threshold
        lte_dl["rb_high_usage_threshold"] = rb_high_threshold
        lte_dl["rb_low_usage_flag"] = rb_values.notna() & (rb_values <= rb_low_threshold)
        lte_dl["rb_medium_or_high_usage_flag"] = rb_values.notna() & (rb_values >= rb_medium_threshold)
        lte_dl["rb_high_usage_flag"] = rb_values.notna() & (rb_values >= rb_high_threshold)
        lte_dl["rb_usage_bucket"] = np.select(
            [
                rb_values.isna(),
                rb_values <= rb_low_threshold,
                rb_values < rb_medium_threshold,
                rb_values < rb_high_threshold,
                rb_values >= rb_high_threshold,
            ],
            ["missing", "low_rb_usage", "below_median_rb_usage", "medium_rb_usage", "high_rb_usage"],
            default="unknown",
        )
        lte_dl["rb_demand_confidence"] = np.select(
            [
                lte_dl["rb_high_usage_flag"],
                lte_dl["rb_medium_or_high_usage_flag"],
                lte_dl["rb_low_usage_flag"],
            ],
            ["high", "medium", "low_or_uncertain"],
            default="unknown",
        )

        # Engineering-normalized RB utilization estimate for future ML/RCA.
        # When aggregate bandwidth is available, use it. Otherwise estimate capacity from
        # serving bandwidth and carrier count. This remains an estimate, but it is more
        # portable than raw RB quantile buckets alone.
        def _lte_rb_capacity_from_bandwidth_mhz(series: pd.Series) -> pd.Series:
            bw = pd.to_numeric(series, errors="coerce")
            return np.select(
                [bw <= 1.4, bw <= 3, bw <= 5, bw <= 10, bw <= 15, bw <= 20],
                [6, 15, 25, 50, 75, 100],
                default=np.nan,
            )

        agg_bw = pd.to_numeric(lte_dl.get("lte_agg_bandwidth_dl", pd.Series(np.nan, index=lte_dl.index)), errors="coerce")
        pcell_bw = pd.to_numeric(lte_dl.get("lte_bandwidth_dl", pd.Series(np.nan, index=lte_dl.index)), errors="coerce")
        carriers = pd.to_numeric(lte_dl.get("carrier_count", pd.Series(1, index=lte_dl.index)), errors="coerce").fillna(1).clip(lower=1)

        estimated_rb_from_agg_bw = pd.Series(_lte_rb_capacity_from_bandwidth_mhz(agg_bw), index=lte_dl.index)
        estimated_rb_from_pcell_bw = pd.Series(_lte_rb_capacity_from_bandwidth_mhz(pcell_bw), index=lte_dl.index) * carriers
        fallback_rb_capacity = 100 * carriers  # conservative 20-MHz-per-carrier fallback when bandwidth is unavailable

        lte_dl["estimated_available_rb"] = estimated_rb_from_agg_bw.fillna(estimated_rb_from_pcell_bw).fillna(fallback_rb_capacity)
        lte_dl["rb_capacity_estimation_method"] = np.select(
            [estimated_rb_from_agg_bw.notna(), estimated_rb_from_pcell_bw.notna()],
            ["aggregate_bandwidth", "pcell_bandwidth_times_carriers"],
            default="fallback_100rb_per_carrier",
        )
        lte_dl["rb_utilization_pct"] = (lte_dl["rb_usage_value"] / lte_dl["estimated_available_rb"].replace(0, np.nan) * 100).clip(lower=0, upper=250)

        rb_demand_summary = (
            lte_dl.groupby(["rb_reference_column", "rb_usage_bucket"], dropna=False)
            .agg(
                rows=("actual_lte_dl_throughput", "count"),
                median_rb=("rb_usage_value", "median"),
                median_tp=("actual_lte_dl_throughput", "median"),
                p25_tp=("actual_lte_dl_throughput", lambda ss: ss.quantile(0.25)),
                p75_tp=("actual_lte_dl_throughput", lambda ss: ss.quantile(0.75)),
                pct_below_10=("actual_lte_dl_throughput", lambda ss: (ss < LOW_TP_FIXED_MBPS).mean() * 100),
            )
            .reset_index()
        )
        for c in ["median_rb", "median_tp", "p25_tp", "p75_tp", "pct_below_10"]:
            rb_demand_summary[c] = rb_demand_summary[c].astype(float).round(3)
        save_table(rb_demand_summary, "throughput_by_rb_demand_summary.csv")
        display(rb_demand_summary)

        # Override the earlier relative low-RB flag so it only uses reliable RB evidence.
        lte_dl["low_rb_allocation_flag"] = lte_dl["rb_low_usage_flag"]

        plot_box_by_category(
            lte_dl,
            "rb_usage_bucket",
            "actual_lte_dl_throughput",
            "Throughput by RB Usage / Demand Bucket",
            "LTE DL throughput (Mbps)",
            "05e_throughput_by_rb_usage_bucket.png",
            category_order=["low_rb_usage", "below_median_rb_usage", "medium_rb_usage", "high_rb_usage", "missing"],
        )
    else:
        lte_dl["rb_usage_value"] = np.nan
        lte_dl["rb_low_usage_threshold"] = np.nan
        lte_dl["rb_medium_usage_threshold"] = np.nan
        lte_dl["rb_high_usage_threshold"] = np.nan
        lte_dl["rb_low_usage_flag"] = False
        lte_dl["rb_medium_or_high_usage_flag"] = True  # no reliable RB, so do not block P75/CA purely because RB is unavailable
        lte_dl["rb_high_usage_flag"] = False
        lte_dl["rb_usage_bucket"] = "rb_not_reliable_or_missing"
        lte_dl["rb_demand_confidence"] = "unknown_rb_not_reliable"
        lte_dl["low_rb_allocation_flag"] = False

    if "lte_agg_rb_dl" in rb_cols:
        rb_series = pd.to_numeric(lte_dl["lte_agg_rb_dl"], errors="coerce")
        if rb_series.notna().mean() * 100 >= MIN_RELIABLE_RB_NON_NULL_PCT:
            rb_bins = [0, 10, 25, 50, 100, 150, 250, np.inf]
            rb_labels = ["0-10", "10-25", "25-50", "50-100", "100-150", "150-250", "250+"]
            lte_dl["lte_agg_rb_dl_bin"] = pd.cut(rb_series, bins=rb_bins, labels=rb_labels, include_lowest=True, right=False)
            plot_box_by_category(lte_dl, "lte_agg_rb_dl_bin", "actual_lte_dl_throughput", "Throughput by Aggregate DL RB Usage Bin", "LTE DL throughput (Mbps)", "05c_throughput_by_agg_rb_bin.png", category_order=rb_labels)
            plot_scatter_sample(lte_dl, "lte_agg_rb_dl", "actual_lte_dl_throughput", "Throughput vs Aggregate DL RB Usage", "Aggregate DL RB usage", "LTE DL throughput (Mbps)", "05d_tp_vs_agg_rb.png")

    plot_scatter_sample(lte_dl, "lte_rsrp", "actual_lte_dl_throughput", "Throughput vs RSRP", "RSRP (dBm)", "LTE DL throughput (Mbps)", "06_tp_vs_rsrp.png")
    plot_scatter_sample(lte_dl, "lte_rsrq", "actual_lte_dl_throughput", "Throughput vs RSRQ", "RSRQ (dB)", "LTE DL throughput (Mbps)", "07_tp_vs_rsrq.png")
    plot_scatter_sample(lte_dl, "lte_sinr", "actual_lte_dl_throughput", "Throughput vs SINR", "SINR (dB)", "LTE DL throughput (Mbps)", "08_tp_vs_sinr.png")

    plot_route_metric(
        lte_dl,
        "actual_lte_dl_throughput",
        "Route Colored by LTE DL Throughput Before Anomaly Detection",
        "09_route_by_throughput.png",
        clip_upper_q=ROUTE_MAP_COLOR_CLIP_Q,
        colorbar_label="LTE DL throughput Mbps",
    )


# -------------------------
# Save clean pre-anomaly feature table for the next notebook and future ML
# -------------------------
# This table is intentionally exported BEFORE any anomaly labels, expected-throughput P75
# references, residuals, scores, or severity labels are created. It is the leakage-safe base
# for Notebook 02 and, later, for ML expected-throughput modeling.
forbidden_pre_anomaly_prefixes = (
    "expected_tp", "throughput_gap_p75", "throughput_ratio_p75", "log_residual_p75",
    "throughput_gap_rb_p75", "throughput_ratio_rb_p75", "log_residual_rb_p75",
    "score_", "anomaly_score", "anomaly_type", "anomaly_severity", "is_throughput_anomaly",
    "underperform_", "low_tp_", "rolling_", "prev_tp", "tp_drop_", "tp_ratio_to_prev",
    "bad_session", "global_p10", "global_p25", "session_p10", "session_p25",
)
forbidden_pre_anomaly_exact = {
    "anomaly_method_count", "context_flag_count", "independent_trigger_count", "trigger_family_flags",
    "rca_handoff_flag", "rca_status", "rca_decision_placeholder",
}
forbidden_cols = [
    c for c in lte_dl.columns
    if c in forbidden_pre_anomaly_exact or any(str(c).startswith(prefix) for prefix in forbidden_pre_anomaly_prefixes)
]
pre_anomaly_feature_table = lte_dl.drop(columns=forbidden_cols, errors="ignore").copy()

pre_anomaly_leakage_check = pd.DataFrame({
    "removed_if_present": forbidden_cols,
    "reason": "post-anomaly/statistical-score/expected-throughput leakage guard",
})
save_table(pre_anomaly_leakage_check, "pre_anomaly_feature_table_leakage_guard.csv", category="preprocessing")

remaining_forbidden_cols = [
    c for c in pre_anomaly_feature_table.columns
    if c in forbidden_pre_anomaly_exact or any(str(c).startswith(prefix) for prefix in forbidden_pre_anomaly_prefixes)
]
pre_anomaly_leakage_validation_summary = pd.DataFrame([{
    "input_columns": int(lte_dl.shape[1]),
    "forbidden_columns_found": int(len(forbidden_cols)),
    "forbidden_columns_removed": int(len(forbidden_cols)),
    "output_columns": int(pre_anomaly_feature_table.shape[1]),
    "forbidden_columns_remaining": int(len(remaining_forbidden_cols)),
    "validation_status": "PASS" if not remaining_forbidden_cols else "FAIL",
    "remaining_forbidden_column_names": ";".join(map(str, remaining_forbidden_cols)),
}])
save_table(
    pre_anomaly_leakage_validation_summary,
    "pre_anomaly_leakage_guard_validation_summary.csv",
    category="preprocessing",
)
if remaining_forbidden_cols:
    raise AssertionError(f"Post-anomaly leakage columns remain in the pre-anomaly table: {remaining_forbidden_cols[:20]}")

# Machine-readable cadence validation. The project expectation is one sample per second.
if "sample_gap_seconds" in pre_anomaly_feature_table.columns:
    _gap = pd.to_numeric(pre_anomaly_feature_table["sample_gap_seconds"], errors="coerce")
else:
    _ts = pd.to_datetime(pre_anomaly_feature_table.get("timestamp"), errors="coerce", utc=True)
    _grp = pre_anomaly_feature_table.get("id", pd.Series("all", index=pre_anomaly_feature_table.index))
    _gap = _ts.groupby(_grp, dropna=False).diff().dt.total_seconds()
_gap_valid = _gap[_gap.notna() & (_gap >= 0)]
sampling_gap_validation_summary = pd.DataFrame([{
    "rows": int(len(pre_anomaly_feature_table)),
    "sessions": int(pre_anomaly_feature_table["id"].nunique()) if "id" in pre_anomaly_feature_table.columns else np.nan,
    "valid_gap_rows": int(_gap_valid.size),
    "median_gap_seconds": float(_gap_valid.median()) if len(_gap_valid) else np.nan,
    "p90_gap_seconds": float(_gap_valid.quantile(0.90)) if len(_gap_valid) else np.nan,
    "p99_gap_seconds": float(_gap_valid.quantile(0.99)) if len(_gap_valid) else np.nan,
    "pct_exactly_1_second": float((_gap_valid.sub(SAMPLE_INTERVAL_SECONDS).abs() <= 1e-9).mean() * 100) if len(_gap_valid) else np.nan,
    "pct_within_1_5_seconds": float((_gap_valid <= 1.5 * SAMPLE_INTERVAL_SECONDS).mean() * 100) if len(_gap_valid) else np.nan,
    "pct_above_3_seconds": float((_gap_valid > 3.0).mean() * 100) if len(_gap_valid) else np.nan,
    "expected_sample_interval_seconds": SAMPLE_INTERVAL_SECONDS,
    "validation_status": "PASS" if len(_gap_valid) and abs(float(_gap_valid.median()) - SAMPLE_INTERVAL_SECONDS) <= 0.01 else "REVIEW",
}])
save_table(sampling_gap_validation_summary, "sampling_gap_validation_summary.csv", category="preprocessing")

save_parquet_or_csv(pre_anomaly_feature_table, "lte_dl_feature_table_clean_pre_anomaly", category="preprocessing")
# Backward-compatible alias. In this split version, this name now points to the clean table,
# not the post-scoring anomaly audit table.
save_parquet_or_csv(pre_anomaly_feature_table, "lte_dl_modeling_table", category="preprocessing")

print("Saved leakage-safe clean pre-anomaly feature table:", PREPROCESSING_DIR / "lte_dl_feature_table_clean_pre_anomaly.parquet")
print("Clean pre-anomaly shape:", pre_anomaly_feature_table.shape)


Saved CSV: Throughput_Anomaly_Detection_Outputs\02_anomaly_detection\throughput_threshold_sanity.csv  shape=(15, 2)


,metric,value
0,rows,32238.000
1,throughput_p01,0.016
2,throughput_p05,1.321
3,throughput_p10,4.990
4,throughput_p17,9.856
5,throughput_p25,15.431
6,throughput_median,38.801
7,throughput_p75,74.473
8,throughput_p90,115.824
9,throughput_p95,139.674


Saved interactive plot: Throughput_Anomaly_Detection_Outputs\03_plots\02b_sample_gap_seconds_distribution.html
Saved static companion image: Throughput_Anomaly_Detection_Outputs\03_plots\02b_sample_gap_seconds_distribution.png


Saved interactive plot: Throughput_Anomaly_Detection_Outputs\03_plots\02c_throughput_by_download_activity_context.html
Saved static companion image: Throughput_Anomaly_Detection_Outputs\03_plots\02c_throughput_by_download_activity_context.png


Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\throughput_by_download_context_summary.csv  shape=(3, 8)


...

Saved interactive plot: Throughput_Anomaly_Detection_Outputs\03_plots\06_tp_vs_rsrp.html
Saved static companion image: Throughput_Anomaly_Detection_Outputs\03_plots\06_tp_vs_rsrp.png


Saved interactive plot: Throughput_Anomaly_Detection_Outputs\03_plots\07_tp_vs_rsrq.html
Saved static companion image: Throughput_Anomaly_Detection_Outputs\03_plots\07_tp_vs_rsrq.png


Saved interactive plot: Throughput_Anomaly_Detection_Outputs\03_plots\08_tp_vs_sinr.html
Saved static companion image: Throughput_Anomaly_Detection_Outputs\03_plots\08_tp_vs_sinr.png


Saved interactive plot: Throughput_Anomaly_Detection_Outputs\03_plots\09_route_by_throughput.html
Saved static companion image: Throughput_Anomaly_Detection_Outputs\03_plots\09_route_by_throughput.png


Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\pre_anomaly_feature_table_leakage_guard.csv  shape=(0, 2)
Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\pre_anomaly_leakage_guard_validation_summary.csv  shape=(1, 7)
Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\sampling_gap_validation_summary.csv  shape=(1, 11)
Saved Parquet: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\lte_dl_feature_table_clean_pre_anomaly.parquet  shape=(32238, 353)
Saved Parquet: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\lte_dl_modeling_table.parquet  shape=(32238, 353)
Saved leakage-safe clean pre-anomaly feature table: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\lte_dl_feature_table_clean_pre_anomaly.parquet
Clean pre-anomaly shape: (32238, 353)


In [14]:
# =========================
# Final preprocessing run metadata
# =========================
def _sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> Optional[str]:
    if not path.exists() or not path.is_file():
        return None
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

_notebook_candidates = [
    Path("10. LTE_Throughput_PreAnomaly_Final_Handoff_Ready.ipynb"),
    Path("10. LTE_Throughput_PreAnomaly_Final_Optimized.ipynb"),
]
_existing_nb = next((p for p in _notebook_candidates if p.exists()), None)
_input_candidates = [DRIVE_TEST_PATH]
_existing_input = next((p for p in _input_candidates if p.exists()), None)

preprocessing_run_metadata = {
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "stage": "pre_anomaly_feature_engineering",
    "pipeline_version": "final_handoff_ready_v1",
    "clean_generated_outputs_before_run": bool(CLEAN_GENERATED_OUTPUTS_BEFORE_RUN),
    "notebook_file": str(_existing_nb) if _existing_nb else None,
    "notebook_sha256": _sha256_file(_existing_nb) if _existing_nb else None,
    "input_file": str(_existing_input) if _existing_input else None,
    "input_sha256": _sha256_file(_existing_input) if _existing_input else None,
    "output_rows": int(len(pre_anomaly_feature_table)),
    "output_columns": int(pre_anomaly_feature_table.shape[1]),
    "sessions": int(pre_anomaly_feature_table["id"].nunique()) if "id" in pre_anomaly_feature_table.columns else None,
    "generated_file_count_after_notebook_01": int(sum(1 for p in OUTPUT_ROOT.rglob("*") if p.is_file())),
}
(PREPROCESSING_DIR / "preprocessing_run_metadata.json").write_text(
    json.dumps(preprocessing_run_metadata, indent=2, default=str),
    encoding="utf-8",
)
display(pd.DataFrame([preprocessing_run_metadata]))


,run_timestamp_utc,stage,pipeline_version,clean_generated_outputs_before_run,notebook_file,notebook_sha256,input_file,input_sha256,output_rows,output_columns,sessions,generated_file_count_after_notebook_01
0,2026-08-04T16:07:23.945450+00:00,pre_anomaly_feature_engineering,final_handoff_ready_v1,True,None,None,Raw_Data\Network_Drive_Test.pkl,57c34f4b4d33f86216fc75c92af95ef4b64f99e4aa1aacbde28065040076c07b,32238,353,271,63


In [15]:
# =========================
# Final preprocessing output index
# =========================
output_index = []
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        output_index.append({
            "file": path.name,
            "folder": str(path.parent.relative_to(OUTPUT_ROOT)),
            "size": human_bytes(path.stat().st_size),
            "path": str(path),
        })
output_index_df = pd.DataFrame(output_index)
save_table(output_index_df, "output_index.csv", category="preprocessing")
display(output_index_df)

print("Notebook 01 complete.")
print("Clean pre-anomaly table:", PREPROCESSING_DIR / "lte_dl_feature_table_clean_pre_anomaly.parquet")
print("Next step: run Notebook 02 using this table.")

Saved CSV: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\output_index.csv  shape=(64, 4)


,file,folder,size,path
0,activity_table.csv,01_preprocessing_cleaning,235.32 KB,Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\activity_table.csv
1,activity_type_summary.csv,01_preprocessing_cleaning,56.00 B,Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\activity_type_summary.csv
2,canonical_column_mapping.csv,01_preprocessing_cleaning,1.62 KB,Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\canonical_column_mapping.csv
3,cleaned_throughput_data_dictionary.csv,01_preprocessing_cleaning,57.26 KB,Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\cleaned_throughput_data_dictionary.csv
4,cleaning_summary.csv,01_preprocessing_cleaning,180.00 B,Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\cleaning_summary.csv
5,column_family_summary_after_cleaning.csv,01_preprocessing_cleaning,1.14 KB,Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\column_family_summary_after_cleaning.csv
6,column_family_summary_before_cleaning.csv,01_preprocessing_cleaning,738.00 B,Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\column_family_summary_before_cleaning.csv
7,download_activity_intervals_used.csv,01_preprocessing_cleaning,46.55 KB,Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\download_activity_intervals_used.csv
8,download_activity_window_summary.csv,01_preprocessing_cleaning,176.00 B,Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\download_activity_window_summary.csv
9,drive_clean_throughput_base.csv.gz,01_preprocessing_cleaning,13.21 MB,Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\drive_clean_throughput_base.csv.gz


Notebook 01 complete.
Clean pre-anomaly table: Throughput_Anomaly_Detection_Outputs\01_preprocessing_cleaning\lte_dl_feature_table_clean_pre_anomaly.parquet
Next step: run Notebook 02 using this table.


# Notebook 10 completion checklist

Before continuing to Notebook 11, confirm:

- the notebook completed without errors;
- `pre_anomaly_leakage_guard_validation_summary.csv` reports PASS;
- `sampling_gap_validation_summary.csv` reports PASS;
- `lte_dl_feature_table_clean_pre_anomaly.parquet` and `lte_dl_modeling_table.parquet` exist;
- the generated-output folder contains only files from this run.

The symmetric event-window features remain offline RCA context only. Causal ML views in Notebook 11 use past-only event and throughput history.
